In [ ]:
# ============================================================
# ANOMALY INJECTION — PREPARE TRAIN / VALIDATION / TEST DATA
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path


# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

TRAIN_PATH = Path(
    "/home/jetson/Desktop/ciesol-anomaly-detection/data/processed/train_unscaled.parquet"
)

VALIDATION_PATH = Path(
    "/home/jetson/Desktop/ciesol-anomaly-detection/data/processed/validation_unscaled.parquet"
)

TEST_PATH = Path(
    "/home/jetson/Desktop/ciesol-anomaly-detection/data/processed/test_unscaled.parquet"
)


# ------------------------------------------------------------
# 2. Check that all files exist
# ------------------------------------------------------------

for path, name in [
    (TRAIN_PATH, "Train"),
    (VALIDATION_PATH, "Validation"),
    (TEST_PATH, "Test"),
]:
    if not path.exists():
        raise FileNotFoundError(
            f"{name} file not found: {path}"
        )


# ------------------------------------------------------------
# 3. Load CLEAN, UNSCALED data
# ------------------------------------------------------------

df_train = pd.read_parquet(TRAIN_PATH)
df_validation = pd.read_parquet(VALIDATION_PATH)
df_test = pd.read_parquet(TEST_PATH)


print("=" * 70)
print("CLEAN DATASETS LOADED")
print("=" * 70)

print(f"\nTrain      : {df_train.shape}")
print(f"Validation : {df_validation.shape}")
print(f"Test       : {df_test.shape}")


# ------------------------------------------------------------
# 4. Convert timestamps
# ------------------------------------------------------------

for name, df in [
    ("Train", df_train),
    ("Validation", df_validation),
    ("Test", df_test),
]:

    df["FECHA"] = pd.to_datetime(
        df["FECHA"],
        errors="coerce"
    )

    if df["FECHA"].isna().any():
        raise ValueError(
            f"{name} contains invalid timestamps in 'FECHA'."
        )


# ------------------------------------------------------------
# 5. Ensure chronological order
# ------------------------------------------------------------

df_train = (
    df_train
    .sort_values("FECHA")
    .reset_index(drop=True)
)

df_validation = (
    df_validation
    .sort_values("FECHA")
    .reset_index(drop=True)
)

df_test = (
    df_test
    .sort_values("FECHA")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 6. Sensor columns used by the professor's method
# ------------------------------------------------------------

SENSORS = [
    "PCO2EXT",
    "PHEXT",
    "PRAD",
    "PRGINT",
    "PTEXT",
    "PVV",
    "XCO2I",
    "XHINV",
    "XTINV",
]

VENTILATION_COLUMNS = [
    "UVENT_cen",
    "UVENT_lN",
]


# ------------------------------------------------------------
# 7. Required columns
# ------------------------------------------------------------

required_columns = (
    ["FECHA"]
    + SENSORS
    + VENTILATION_COLUMNS
)


for name, df in [
    ("Train", df_train),
    ("Validation", df_validation),
    ("Test", df_test),
]:

    missing_columns = [
        col
        for col in required_columns
        if col not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{name} is missing required columns: "
            f"{missing_columns}"
        )


# ------------------------------------------------------------
# 8. Keep completely clean reference copies
# ------------------------------------------------------------

df_train_clean = df_train.copy()

df_validation_clean = df_validation.copy()

df_test_clean = df_test.copy()


# ------------------------------------------------------------
# 9. Missing-value check BEFORE injection
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MISSING VALUES BEFORE INJECTION")
print("=" * 70)

for name, df in [
    ("TRAIN", df_train),
    ("VALIDATION", df_validation),
    ("TEST", df_test),
]:

    print(f"\n{name}")

    missing_counts = (
        df[SENSORS]
        .isna()
        .sum()
    )

    print(missing_counts.to_string())


# ------------------------------------------------------------
# 10. Prepare independent injection copies
# ------------------------------------------------------------

# These are the datasets that will actually receive anomalies.

df_train_injected = df_train.copy()

df_validation_injected = df_validation.copy()

df_test_eval = df_test.copy()


# ------------------------------------------------------------
# 11. Injection traceability columns
# ------------------------------------------------------------

INJECTION_COLUMNS = [
    "sensor_anomalia",
    "anomalia_id",
    "tipo_anomalia",
    "fuente_anomalia",
    "valor_original_anomalia",
    "valor_inyectado_anomalia",
]


def initialize_injection_columns(df):
    """Add clean anomaly-traceability columns."""

    df["sensor_anomalia"] = pd.NA
    df["anomalia_id"] = pd.NA
    df["tipo_anomalia"] = pd.NA
    df["fuente_anomalia"] = pd.NA

    df["valor_original_anomalia"] = np.nan
    df["valor_inyectado_anomalia"] = np.nan

    return df


df_train_injected = initialize_injection_columns(
    df_train_injected
)

df_validation_injected = initialize_injection_columns(
    df_validation_injected
)

df_test_eval = initialize_injection_columns(
    df_test_eval
)


# ------------------------------------------------------------
# 12. Safety checks BEFORE injection
# ------------------------------------------------------------

for name, df_injected, df_clean in [
    ("Train", df_train_injected, df_train_clean),
    ("Validation", df_validation_injected, df_validation_clean),
    ("Test", df_test_eval, df_test_clean),
]:

    if len(df_injected) != len(df_clean):
        raise RuntimeError(
            f"{name}: unexpected row-count change "
            "before injection."
        )

    if not df_injected["FECHA"].equals(
        df_clean["FECHA"]
    ):
        raise RuntimeError(
            f"{name}: timestamp order changed "
            "unexpectedly before injection."
        )


# ------------------------------------------------------------
# 13. Dataset periods
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DATASET PERIODS")
print("=" * 70)

for name, df in [
    ("TRAIN", df_train),
    ("VALIDATION", df_validation),
    ("TEST", df_test),
]:

    print(
        f"{name:12s}: "
        f"{df['FECHA'].min()} "
        f"-> "
        f"{df['FECHA'].max()}"
    )


# ------------------------------------------------------------
# 14. Final preparation summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ANOMALY INJECTION PREPARATION COMPLETED")
print("=" * 70)

print(f"\nTrain rows      : {len(df_train_injected):,}")
print(f"Validation rows : {len(df_validation_injected):,}")
print(f"Test rows       : {len(df_test_eval):,}")

print(f"\nSensors for injection       : {len(SENSORS)}")
print(f"Ventilation/context vars   : {len(VENTILATION_COLUMNS)}")

print("\nInjection datasets:")
print("  df_train_injected       → training anomalies")
print("  df_validation_injected  → validation anomalies")
print("  df_test_eval            → test evaluation anomalies")

print("\nClean references:")
print("  df_train_clean")
print("  df_validation_clean")
print("  df_test_clean")

print("\nData scale: physical / unscaled")
print("No anomalies injected yet.")
print("No original dataset has been modified.")

In [ ]:
# ============================================================
# SENSOR CONFIGURATION FOR ANOMALY INJECTION
# ============================================================

# Sensors that will receive synthetic anomalies.
# This follows the professor's injection methodology.
SENSORS_INJECTION = [
    "PCO2EXT",
    "PHEXT",
    "PRAD",
    "PRGINT",
    "PTEXT",
    "PVV",
    "XCO2I",
    "XHINV",
    "XTINV",
]

# Ventilation variables are contextual variables.
# They are NOT direct anomaly-injection targets.
VENTILATION_COLUMNS = [
    "UVENT_cen",
    "UVENT_lN",
]

# XTS is intentionally excluded from injection targets.
EXCLUDED_SENSORS = [
    "XTS",
]


# ------------------------------------------------------------
# Verify required columns in ALL clean datasets
# ------------------------------------------------------------

datasets_clean = {
    "TRAIN": df_train_clean,
    "VALIDATION": df_validation_clean,
    "TEST": df_test_clean,
}

required_injection_columns = (
    SENSORS_INJECTION
    + VENTILATION_COLUMNS
)

for dataset_name, df in datasets_clean.items():

    missing_columns = [
        col
        for col in required_injection_columns
        if col not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{dataset_name}: missing required columns: "
            f"{missing_columns}"
        )


# ------------------------------------------------------------
# Original statistics
# Calculated BEFORE synthetic anomaly injection.
#
# Keep statistics separate by split.
# ------------------------------------------------------------

std_devs_original = {
    dataset_name: {
        sensor: df[sensor].std()
        for sensor in SENSORS_INJECTION
    }
    for dataset_name, df in datasets_clean.items()
}


# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("=" * 70)
print("SENSOR CONFIGURATION")
print("=" * 70)

print("\nSensors used for anomaly injection:")
print(SENSORS_INJECTION)

print(f"\nNumber of injection sensors: {len(SENSORS_INJECTION)}")

print("\nContext variables:")
print(VENTILATION_COLUMNS)

print("\nExcluded from direct injection:")
print(EXCLUDED_SENSORS)


print("\n" + "=" * 70)
print("ORIGINAL SENSOR STANDARD DEVIATIONS")
print("=" * 70)

for dataset_name, sensor_stats in std_devs_original.items():

    print(f"\n{dataset_name}")

    for sensor, std in sensor_stats.items():
        print(f"  {sensor:12s}: {std:.4f}")


print("\n" + "=" * 70)
print("CONFIGURATION CHECK")
print("=" * 70)

print("✓ 9 sensors configured for anomaly injection")
print("✓ 2 ventilation/context variables")
print("✓ XTS excluded from direct injection")
print("✓ Required columns verified in Train")
print("✓ Required columns verified in Validation")
print("✓ Required columns verified in Test")
print("✓ Original statistics calculated before injection")

In [ ]:
# ============================================================
# SENSOR ATASCADO — INJECTION
# ============================================================
#
# Applied independently to:
#   - TRAIN
#   - VALIDATION
#   - TEST EVALUATION COPY
#
# The original clean datasets remain untouched.
#
# Methodology:
#   - Same professor injection method
#   - 4 sequences per stuck sensor
#   - Random seed = 42
#   - No anomaly overlap within each dataset
#
# Traceability:
#   - sensor_anomalia
#   - anomalia_id
#   - tipo_anomalia
#   - fuente_anomalia
#   - valor_original_anomalia
#   - valor_inyectado_anomalia
# ============================================================

import hashlib
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

RANDOM_STATE = 42

# 4 sequences per sensor
STUCK_EVENTS_PER_SENSOR = 4

SENSORES_STUCK = [
    "PCO2EXT",
    "PHEXT",
    "PRGINT",
    "PTEXT",
    "PVV",
    "XCO2I",
    "XHINV",
    "XTINV",
]

SENSORES_TEMP_ESPECIALES = {
    "PTEXT",
    "XTINV",
}


# ------------------------------------------------------------
# 2. Sensor-specific duration ranges
# ------------------------------------------------------------

STUCK_DURACION = {
    "PCO2EXT": {"duracion_min": 20, "duracion_max": 40},
    "PTEXT":   {"duracion_min": 16, "duracion_max": 40},
    "XTINV":   {"duracion_min": 20, "duracion_max": 60},
    "XHINV":   {"duracion_min": 28, "duracion_max": 70},
    "PHEXT":   {"duracion_min": 32, "duracion_max": 80},
    "PVV":     {"duracion_min": 20, "duracion_max": 60},
    "XCO2I":   {"duracion_min": 20, "duracion_max": 60},
    "PRGINT":  {"duracion_min": 20, "duracion_max": 60},
}


# ------------------------------------------------------------
# 3. Minimum separation from natural signal
# ------------------------------------------------------------

STUCK_MIN_SIGMA_SEPARACION = {
    "PCO2EXT": 0.50,
    "PHEXT":   0.50,
    "PRGINT":  0.50,
    "PTEXT":   0.50,
    "PVV":     0.50,
    "XCO2I":   0.50,
    "XHINV":   0.50,
    "XTINV":   0.50,
}


# ------------------------------------------------------------
# 4. Physical ranges
# ------------------------------------------------------------

RANGOS_FISICOS_STUCK = {
    "PCO2EXT": {"min": 300, "max": 800},
    "PHEXT":   {"min": 0,   "max": 100},
    "PRGINT":  {"min": 0,   "max": 1200},
    "PTEXT":   {"min": -20, "max": 60},
    "PVV":     {"min": 0,   "max": 50},
    "XCO2I":   {"min": 200, "max": 3000},
    "XHINV":   {"min": 0,   "max": 100},
    "XTINV":   {"min": -5,  "max": 60},
}


# ------------------------------------------------------------
# 5. Temperature-specific conditions
# ------------------------------------------------------------

TEMP_PERIODOS_PERMITIDOS = {
    "morning",
    "afternoon",
}

TEMP_PRAD_MEDIA_MIN = 5.0

TEMP_DELTA_MIN_ABS = {
    "PTEXT": 1.50,
    "XTINV": 1.50,
}

TEMP_DELTA_MAX_ABS = {
    "PTEXT": 4.0,
    "XTINV": 5.0,
}

TEMP_RANGO_MIN_ABS = {
    "PTEXT": 0.75,
    "XTINV": 0.75,
}


# ------------------------------------------------------------
# 6. XCO2I validation
# ------------------------------------------------------------

XCO2I_PREV_OUTLIER_MAX_SIGMA = 1.0
XCO2I_SEV_MAX_SIGMA = 5.0
XCO2I_FRAC_GT3_MAX = 0.10
XCO2I_FRAC_DEBIL_MAX = 0.60
XCO2I_PREV_WINDOW = 10


# ------------------------------------------------------------
# 7. Helper: deterministic seed
# ------------------------------------------------------------

def stable_seed(base_seed, *parts):
    """
    Generate a deterministic seed for each sensor/event selection.
    """

    text = "|".join(
        [str(base_seed), *map(str, parts)]
    )

    digest = hashlib.blake2b(
        text.encode("utf-8"),
        digest_size=8,
    ).digest()

    return int.from_bytes(
        digest,
        "little"
    ) % (2**32)


def periodo_dia(hora):

    if 6 <= hora < 12:
        return "morning"

    if 12 <= hora < 18:
        return "afternoon"

    if 18 <= hora < 22:
        return "evening"

    return "night"


# ------------------------------------------------------------
# 8. Validate XCO2I candidate
# ------------------------------------------------------------

def validar_xco2i_evento(
    df_ref,
    sensor,
    start,
    end,
    prev_value,
    std_devs,
):
    """
    Reject XCO2I stuck events that would themselves look like
    an extreme natural outlier.
    """

    sigma = std_devs.get(
        sensor,
        np.nan
    )

    if (
        sensor != "XCO2I"
        or not np.isfinite(sigma)
        or sigma <= 0
    ):
        return True

    valores = pd.to_numeric(
        df_ref.iloc[start:end][sensor],
        errors="coerce",
    ).to_numpy(dtype=float)

    if (
        len(valores) == 0
        or not np.isfinite(valores).all()
        or not np.isfinite(prev_value)
    ):
        return False

    # Previous local window
    prev_start = max(
        0,
        start - XCO2I_PREV_WINDOW
    )

    prev_window = pd.to_numeric(
        df_ref.iloc[prev_start:start][sensor],
        errors="coerce",
    ).dropna().to_numpy(dtype=float)

    if len(prev_window) >= 3:

        prev_median = float(
            np.median(prev_window)
        )

        if (
            abs(
                float(prev_value)
                - prev_median
            ) / sigma
            > XCO2I_PREV_OUTLIER_MAX_SIGMA
        ):
            return False

    severity = (
        np.abs(
            valores
            - float(prev_value)
        ) / sigma
    )

    if (
        float(np.max(severity))
        > XCO2I_SEV_MAX_SIGMA
    ):
        return False

    if (
        float(np.mean(severity > 3.0))
        > XCO2I_FRAC_GT3_MAX
    ):
        return False

    if (
        float(np.mean(severity < 0.5))
        > XCO2I_FRAC_DEBIL_MAX
    ):
        return False

    return True


# ------------------------------------------------------------
# 9. Candidate generation
# ------------------------------------------------------------

def candidatos_stuck(
    df,
    sensor,
    std_devs,
    idx_inicio=0,
    idx_fin=None,
):
    """
    Find possible starting positions for a stuck-sensor event.

    This is the same methodology as the original TRAIN-only
    implementation, but generalized to any dataset.
    """

    if idx_fin is None:
        idx_fin = len(df)

    serie = pd.to_numeric(
        df[sensor],
        errors="coerce"
    )

    valores = serie.to_numpy(
        dtype=float
    )

    previos = np.roll(
        valores,
        1
    )

    previos[0] = np.nan

    cfg = STUCK_DURACION[sensor]

    dur_min = int(
        cfg["duracion_min"]
    )

    dur_max = int(
        cfg["duracion_max"]
    )

    sigma = std_devs.get(
        sensor,
        0.0
    )

    min_sigma = (
        STUCK_MIN_SIGMA_SEPARACION[sensor]
    )

    separation_threshold = max(
        min_sigma * sigma
        if sigma > 0 else 0.0,
        1e-6,
    )

    idx_inicio = max(
        0,
        int(idx_inicio)
    )

    idx_fin = min(
        len(df),
        int(idx_fin)
    )

    max_start = (
        idx_fin
        - dur_min
    )

    if max_start < idx_inicio:

        return pd.DataFrame(
            columns=[
                "start",
                "dur_min",
                "dur_max",
                "period",
            ]
        )

    # --------------------------------------------------------
    # Basic validity
    # --------------------------------------------------------

    valido = (
        np.isfinite(valores)
        & np.isfinite(previos)
        & ~np.isclose(
            valores,
            previos,
            rtol=0.0,
            atol=1e-12,
        )
    )

    # --------------------------------------------------------
    # Local variability for non-temperature sensors
    # --------------------------------------------------------

    if sensor not in SENSORES_TEMP_ESPECIALES:

        rolling_std = (
            serie
            .rolling(
                30,
                min_periods=10
            )
            .std()
            .to_numpy(dtype=float)
        )

        positive_std = rolling_std[
            np.isfinite(rolling_std)
            & (rolling_std > 0)
        ]

        variability_threshold = (
            float(
                np.percentile(
                    positive_std,
                    20
                )
            )
            if len(positive_std)
            else 0.0
        )

        valido &= (
            np.isfinite(rolling_std)
            & (
                rolling_std
                >= variability_threshold
            )
        )

    # --------------------------------------------------------
    # Physical range
    # --------------------------------------------------------

    if sensor in RANGOS_FISICOS_STUCK:

        r = RANGOS_FISICOS_STUCK[sensor]

        valido &= (
            (valores >= r["min"])
            & (valores <= r["max"])
            & (previos >= r["min"])
            & (previos <= r["max"])
        )

    # --------------------------------------------------------
    # Find valid consecutive windows
    # --------------------------------------------------------

    cumulative = np.concatenate(
        [
            [0],
            np.cumsum(
                valido.astype(np.int32)
            ),
        ]
    )

    starts = np.arange(
        idx_inicio,
        max_start + 1,
    )

    window_counts = (
        cumulative[
            starts + dur_min
        ]
        - cumulative[starts]
    )

    starts = starts[
        window_counts == dur_min
    ]

    # --------------------------------------------------------
    # Detailed contextual filtering
    # --------------------------------------------------------

    valid_starts = []

    for start in starts:

        stuck_value = previos[start]

        window = valores[
            start:start + dur_min
        ]

        if (
            len(window) < dur_min
            or not np.isfinite(stuck_value)
            or not np.isfinite(window).all()
        ):
            continue

        # ----------------------------------------------------
        # Temperature sensors
        # ----------------------------------------------------

        if sensor in SENSORES_TEMP_ESPECIALES:

            timestamp = pd.to_datetime(
                df.loc[start, "FECHA"]
            )

            period = periodo_dia(
                timestamp.hour
            )

            if (
                period
                not in TEMP_PERIODOS_PERMITIDOS
            ):
                continue

            if "PRAD" in df.columns:

                prad = pd.to_numeric(
                    df.loc[
                        start:
                        start + dur_min - 1,
                        "PRAD"
                    ],
                    errors="coerce",
                ).to_numpy(dtype=float)

                if (
                    len(prad) < dur_min
                    or not np.isfinite(prad).all()
                    or float(np.mean(prad))
                    < TEMP_PRAD_MEDIA_MIN
                ):
                    continue

            differences = np.abs(
                window - stuck_value
            )

            diff_mean = float(
                np.mean(differences)
            )

            diff_max = float(
                np.max(differences)
            )

            window_range = float(
                np.max(window)
                - np.min(window)
            )

            delta_min = (
                TEMP_DELTA_MIN_ABS[sensor]
            )

            delta_max = (
                TEMP_DELTA_MAX_ABS[sensor]
            )

            range_min = (
                TEMP_RANGO_MIN_ABS[sensor]
            )

            if (
                diff_mean < delta_min
                and diff_max
                < 1.5 * delta_min
            ):
                continue

            if diff_max > delta_max:
                continue

            if (
                window_range < range_min
                and abs(
                    float(window[-1])
                    - float(stuck_value)
                ) < delta_min
            ):
                continue

        # ----------------------------------------------------
        # General sensors
        # ----------------------------------------------------

        else:

            diff_median = float(
                np.median(
                    np.abs(
                        window
                        - stuck_value
                    )
                )
            )

            diff_mean = float(
                abs(
                    np.mean(window)
                    - stuck_value
                )
            )

            diff_max = float(
                np.max(
                    np.abs(
                        window
                        - stuck_value
                    )
                )
            )

            if (
                diff_median
                < separation_threshold
                and diff_mean
                < separation_threshold
                and diff_max
                < 2.0
                * separation_threshold
            ):
                continue

        valid_starts.append(start)

    # --------------------------------------------------------
    # Return candidates
    # --------------------------------------------------------

    if not valid_starts:

        return pd.DataFrame(
            columns=[
                "start",
                "dur_min",
                "dur_max",
                "period",
            ]
        )

    result = pd.DataFrame({
        "start": valid_starts,
    })

    result["dur_min"] = dur_min
    result["dur_max"] = dur_max

    timestamps = pd.to_datetime(
        df.loc[
            result["start"],
            "FECHA"
        ]
    )

    result["period"] = (
        timestamps.dt.hour
        .map(periodo_dia)
        .to_numpy()
    )

    return result


# ------------------------------------------------------------
# 10. Generate candidates for a dataset
# ------------------------------------------------------------

def generate_stuck_candidates(
    df,
    std_devs,
):
    """
    Generate Sensor Atascado candidates independently
    for one dataset.
    """

    candidates_cache = {}
    candidate_summary = []

    for sensor in SENSORES_STUCK:

        candidates = candidatos_stuck(
            df,
            sensor,
            std_devs,
            0,
            len(df),
        )

        candidates_cache[sensor] = candidates

        candidate_summary.append({
            "sensor": sensor,
            "candidates": len(candidates),
        })

    return (
        candidates_cache,
        pd.DataFrame(candidate_summary)
    )


# ------------------------------------------------------------
# 11. Generic Sensor Atascado injection
# ------------------------------------------------------------

def inject_stuck(
    df,
    df_clean,
    candidates_cache,
    std_devs,
    events_per_sensor=4,
    random_state=42,
    dataset_name="DATASET",
):
    """
    Inject non-overlapping Sensor Atascado anomalies into
    one dataset.

    The input df should already contain the traceability
    columns initialized by the preparation cell.
    """

    df_mod = df.copy()

    # Tracks rows already occupied by an anomaly
    used = np.zeros(
        len(df_mod),
        dtype=bool,
    )

    records = []

    anomaly_counter = 1

    for sensor in SENSORES_STUCK:

        candidates = candidates_cache.get(
            sensor,
            pd.DataFrame()
        )

        if candidates.empty:

            print(
                f"{dataset_name} | "
                f"{sensor}: no candidates."
            )

            continue

        rng = np.random.default_rng(
            stable_seed(
                random_state,
                dataset_name,
                "stuck",
                sensor,
            )
        )

        starts = rng.permutation(
            candidates["start"]
            .to_numpy(dtype=int)
        )

        created = 0
        rejected_xco2i = 0

        for start in starts:

            if created >= events_per_sensor:
                break

            candidate = candidates[
                candidates["start"].eq(start)
            ].iloc[0]

            dur_min = int(
                candidate["dur_min"]
            )

            dur_max = int(
                candidate["dur_max"]
            )

            duration = int(
                rng.integers(
                    dur_min,
                    dur_max + 1,
                )
            )

            end = start + duration

            if end > len(df_mod):
                continue

            # ------------------------------------------------
            # No anomaly overlap
            # ------------------------------------------------

            if used[start:end].any():
                continue

            # ------------------------------------------------
            # Original values BEFORE injection
            # ------------------------------------------------

            values = pd.to_numeric(
                df_clean.iloc[
                    start:end
                ][sensor],
                errors="coerce",
            ).to_numpy(dtype=float)

            previous_value = (
                float(
                    pd.to_numeric(
                        df_clean.iloc[start - 1][sensor],
                        errors="coerce",
                    )
                )
                if start > 0
                else np.nan
            )

            if (
                not np.isfinite(previous_value)
                or not np.isfinite(values).all()
            ):
                continue

            # ------------------------------------------------
            # Special XCO2I validation
            # ------------------------------------------------

            if sensor == "XCO2I":

                valid_xco2i = (
                    validar_xco2i_evento(
                        df_clean,
                        sensor,
                        start,
                        end,
                        previous_value,
                        std_devs,
                    )
                )

                if not valid_xco2i:

                    rejected_xco2i += 1
                    continue

            # ------------------------------------------------
            # Save original values
            # ------------------------------------------------

            original_values = values.copy()

            injected_values = np.full(
                len(original_values),
                previous_value,
                dtype=float,
            )

            # ------------------------------------------------
            # Inject stuck value
            # ------------------------------------------------

            df_mod.loc[
                start:end - 1,
                sensor
            ] = previous_value

            # ------------------------------------------------
            # Traceability
            # ------------------------------------------------

            anomaly_id = (
                f"SA_{dataset_name[:3].upper()}_"
                f"{anomaly_counter:06d}"
            )

            df_mod.loc[
                start:end - 1,
                "tipo_anomalia"
            ] = "Sensor Atascado"

            df_mod.loc[
                start:end - 1,
                "sensor_anomalia"
            ] = sensor

            df_mod.loc[
                start:end - 1,
                "anomalia_id"
            ] = anomaly_id

            df_mod.loc[
                start:end - 1,
                "fuente_anomalia"
            ] = "professor_method_stuck"

            df_mod.loc[
                start:end - 1,
                "valor_original_anomalia"
            ] = original_values

            df_mod.loc[
                start:end - 1,
                "valor_inyectado_anomalia"
            ] = injected_values

            # ------------------------------------------------
            # Mark rows as used
            # ------------------------------------------------

            used[start:end] = True

            # ------------------------------------------------
            # Event record
            # ------------------------------------------------

            records.append({
                "dataset": dataset_name,
                "anomalia_id": anomaly_id,
                "sensor": sensor,
                "start_idx": start,
                "end_idx": end - 1,
                "duracion": duration,
                "start_time": df_mod.loc[
                    start,
                    "FECHA"
                ],
                "end_time": df_mod.loc[
                    end - 1,
                    "FECHA"
                ],
                "stuck_value": previous_value,
                "rng_seed": stable_seed(
                    random_state,
                    dataset_name,
                    "stuck",
                    sensor,
                ),
            })

            anomaly_counter += 1
            created += 1

        status = (
            "OK"
            if created >= events_per_sensor
            else f"WARNING: {created}/{events_per_sensor}"
        )

        extra = (
            f" | XCO2I rejected: {rejected_xco2i}"
            if sensor == "XCO2I"
            else ""
        )

        print(
            f"{dataset_name:12s} | "
            f"{sensor:8s}: "
            f"{created:2d} sequences "
            f"{status}{extra}"
        )

    return (
        df_mod,
        pd.DataFrame(records),
        used,
    )


# ============================================================
# 12. GENERATE CANDIDATES — THREE DATASETS
# ============================================================

print("=" * 70)
print("GENERATING SENSOR ATASCADO CANDIDATES")
print("=" * 70)


candidates_train, summary_train = (
    generate_stuck_candidates(
        df_train_clean,
        std_devs_original["TRAIN"],
    )
)

candidates_validation, summary_validation = (
    generate_stuck_candidates(
        df_validation_clean,
        std_devs_original["VALIDATION"],
    )
)

candidates_test, summary_test = (
    generate_stuck_candidates(
        df_test_clean,
        std_devs_original["TEST"],
    )
)


print("\nTRAIN candidate summary:")
print(summary_train.to_string(index=False))

print("\nVALIDATION candidate summary:")
print(summary_validation.to_string(index=False))

print("\nTEST candidate summary:")
print(summary_test.to_string(index=False))


# ============================================================
# 13. APPLY SENSOR ATASCADO INDEPENDENTLY
# ============================================================

print("\n" + "=" * 70)
print("INJECTING SENSOR ATASCADO")
print("=" * 70)


# TRAIN
df_train_after_stuck, df_events_stuck_train, used_train = (
    inject_stuck(
        df_train_injected,
        df_train_clean,
        candidates_train,
        std_devs_original["TRAIN"],
        events_per_sensor=STUCK_EVENTS_PER_SENSOR,
        random_state=RANDOM_STATE,
        dataset_name="TRAIN",
    )
)


# VALIDATION
df_validation_after_stuck, df_events_stuck_validation, used_validation = (
    inject_stuck(
        df_validation_injected,
        df_validation_clean,
        candidates_validation,
        std_devs_original["VALIDATION"],
        events_per_sensor=STUCK_EVENTS_PER_SENSOR,
        random_state=RANDOM_STATE,
        dataset_name="VALIDATION",
    )
)


# TEST EVALUATION COPY
df_test_after_stuck, df_events_stuck_test, used_test = (
    inject_stuck(
        df_test_eval,
        df_test_clean,
        candidates_test,
        std_devs_original["TEST"],
        events_per_sensor=STUCK_EVENTS_PER_SENSOR,
        random_state=RANDOM_STATE,
        dataset_name="TEST",
    )
)


# ============================================================
# 14. COMBINE EVENT TABLES
# ============================================================

df_events_stuck = pd.concat(
    [
        df_events_stuck_train,
        df_events_stuck_validation,
        df_events_stuck_test,
    ],
    ignore_index=True,
)


# ============================================================
# 15. RESULTS
# ============================================================

print("\n" + "=" * 70)
print("SENSOR ATASCADO — RESULTS")
print("=" * 70)

for name, events, used in [
    ("TRAIN", df_events_stuck_train, used_train),
    ("VALIDATION", df_events_stuck_validation, used_validation),
    ("TEST", df_events_stuck_test, used_test),
]:

    print(f"\n{name}")
    print("-" * 50)

    print(
        f"Events created : {len(events)}"
    )

    print(
        f"Rows affected  : {used.sum():,}"
    )

    if not events.empty:

        print("\nEvents per sensor:")

        print(
            events[
                "sensor"
            ]
            .value_counts()
            .sort_index()
            .to_string()
        )

        print("\nDuration statistics:")

        print(
            events
            .groupby("sensor")["duracion"]
            .agg(
                ["count", "min", "median", "max"]
            )
            .to_string()
        )


# ============================================================
# 16. AUDIT FUNCTION
# ============================================================

def audit_sensor_atascado(
    df,
    events,
    dataset_name,
):
    """
    Validate Sensor Atascado anomalies for one dataset.

    The audit validates the events that were actually injected.
    It does NOT require a fixed number of events per sensor, because
    the candidate-generation rules may legitimately produce fewer
    valid events for some sensors.
    """

    print("\n" + "=" * 70)
    print(f"{dataset_name} — SENSOR ATASCADO AUDIT")
    print("=" * 70)

    # ========================================================
    # 1. Identify Sensor Atascado rows
    # ========================================================

    mask_stuck = (
        df["tipo_anomalia"]
        .fillna("")
        .eq("Sensor Atascado")
    )

    n_stuck_rows = int(mask_stuck.sum())

    print(
        f"\nRows Sensor Atascado: "
        f"{n_stuck_rows:,}"
    )

    print(
        f"Event records: "
        f"{len(events):,}"
    )

    # --------------------------------------------------------
    # No events
    # --------------------------------------------------------

    if len(events) == 0:

        if n_stuck_rows > 0:
            raise ValueError(
                f"{dataset_name}: Sensor Atascado rows exist "
                "but there are no event records."
            )

        print("\n✓ No Sensor Atascado events.")
        return pd.DataFrame()

    # ========================================================
    # 2. Event table basic checks
    # ========================================================

    if "anomalia_id" not in events.columns:
        raise ValueError(
            f"{dataset_name}: event table has no "
            "'anomalia_id' column."
        )

    if "sensor" not in events.columns:
        raise ValueError(
            f"{dataset_name}: event table has no "
            "'sensor' column."
        )

    duplicate_event_ids = (
        events["anomalia_id"]
        .duplicated()
        .sum()
    )

    print(
        f"\nDuplicate event IDs in event table: "
        f"{duplicate_event_ids:,}"
    )

    if duplicate_event_ids > 0:
        raise ValueError(
            f"{dataset_name}: duplicate anomaly IDs "
            "exist in event table."
        )

    # ========================================================
    # 3. Constant-value audit
    # ========================================================

    audit_rows = []

    subset = df.loc[
        mask_stuck
        & df["sensor_anomalia"].notna()
        & df["anomalia_id"].notna()
    ]

    for (
        sensor,
        anomaly_id
    ), group in subset.groupby(
        [
            "sensor_anomalia",
            "anomalia_id",
        ]
    ):

        # Make sure the sensor column actually exists
        if sensor not in df.columns:
            raise ValueError(
                f"{dataset_name}: sensor '{sensor}' "
                "does not exist in dataframe."
            )

        values = pd.to_numeric(
            group[sensor],
            errors="coerce",
        )

        audit_rows.append({
            "sensor": sensor,
            "anomalia_id": anomaly_id,
            "n_rows": len(group),
            "unique_values": int(
                values.nunique(
                    dropna=True
                )
            ),
            "nan_count": int(
                values.isna().sum()
            ),
        })

    df_audit = pd.DataFrame(audit_rows)

    if not df_audit.empty:

        non_constant = df_audit[
            df_audit["unique_values"] != 1
        ]

        with_nan = df_audit[
            df_audit["nan_count"] > 0
        ]

        if not non_constant.empty:

            print("\nNon-constant events:")
            display(non_constant)

            raise ValueError(
                f"{dataset_name}: some Sensor Atascado "
                "events are not constant."
            )

        if not with_nan.empty:

            print("\nEvents containing NaN:")
            display(with_nan)

            raise ValueError(
                f"{dataset_name}: some Sensor Atascado "
                "events contain NaN."
            )

        print(
            "\n✓ All injected stuck events are constant."
        )

        print(
            "✓ No NaN inside stuck events."
        )

    # ========================================================
    # 4. Traceability columns
    # ========================================================

    traceability_columns = [
        "sensor_anomalia",
        "anomalia_id",
        "tipo_anomalia",
        "fuente_anomalia",
        "valor_original_anomalia",
        "valor_inyectado_anomalia",
    ]

    traceability_ok = True

    print("\nTraceability:")

    for col in traceability_columns:

        if col not in df.columns:

            print(
                f"{col:<30}: COLUMN MISSING"
            )

            traceability_ok = False
            continue

        missing = int(
            df.loc[
                mask_stuck,
                col
            ]
            .isna()
            .sum()
        )

        print(
            f"{col:<30}: "
            f"{missing:,} missing"
        )

        if missing > 0:
            traceability_ok = False

    # ========================================================
    # 5. Verify anomaly type and source
    # ========================================================

    wrong_type = (
        df.loc[
            mask_stuck,
            "tipo_anomalia"
        ]
        .ne("Sensor Atascado")
        .sum()
    )

    wrong_source = (
        df.loc[
            mask_stuck,
            "fuente_anomalia"
        ]
        .ne("professor_method_stuck")
        .sum()
    )

    print(
        f"\nRows with wrong anomaly type: "
        f"{wrong_type:,}"
    )

    print(
        f"Rows with wrong anomaly source: "
        f"{wrong_source:,}"
    )

    if wrong_type > 0:
        traceability_ok = False

    if wrong_source > 0:
        traceability_ok = False

        # ========================================================
    # 6. Verify original vs injected
    # ========================================================
    #
    # For a stuck-sensor event, it is acceptable for some
    # original values to already equal the injected stuck value.
    #
    # What must be true is:
    #   - every event has at least one changed value
    #   - the injected sensor values are constant
    #
    # ========================================================

    original = pd.to_numeric(
        df.loc[
            mask_stuck,
            "valor_original_anomalia"
        ],
        errors="coerce",
    )

    injected = pd.to_numeric(
        df.loc[
            mask_stuck,
            "valor_inyectado_anomalia"
        ],
        errors="coerce",
    )

    comparable = (
        original.notna()
        & injected.notna()
    )

    same_value_count = int(
        np.isclose(
            original.loc[comparable].to_numpy(dtype=float),
            injected.loc[comparable].to_numpy(dtype=float),
            rtol=0.0,
            atol=1e-12,
        ).sum()
    )

    changed_value_count = int(
        comparable.sum() - same_value_count
    )

    print(
        f"\nRows where original == injected: "
        f"{same_value_count:,}"
    )

    print(
        f"Rows where original != injected: "
        f"{changed_value_count:,}"
    )

    # --------------------------------------------------------
    # Event-level check
    # --------------------------------------------------------

    event_change_counts = (
        df.loc[mask_stuck]
        .groupby("anomalia_id")
        .apply(
            lambda g: int(
                np.isclose(
                    pd.to_numeric(
                        g["valor_original_anomalia"],
                        errors="coerce"
                    ).to_numpy(dtype=float),
                    pd.to_numeric(
                        g["valor_inyectado_anomalia"],
                        errors="coerce"
                    ).to_numpy(dtype=float),
                    rtol=0.0,
                    atol=1e-12,
                ).sum()
            )
        )
    )

    event_sizes = (
        df.loc[mask_stuck]
        .groupby("anomalia_id")
        .size()
    )

    event_changed_counts = (
        event_sizes - event_change_counts
    )

    events_without_changes = (
        event_changed_counts[
            event_changed_counts == 0
        ]
    )

    print(
        f"Events without any changed value: "
        f"{len(events_without_changes):,}"
    )

    values_changed_ok = (
        len(events_without_changes) == 0
    )

    print(
        f"Values actually changed: "
        f"{values_changed_ok}"
    )

    if not values_changed_ok:

        print(
            "\nEvents with NO changed values:"
        )
        print(
            events_without_changes
        )

        traceability_ok = False

    # ========================================================
    # 7. Event IDs in dataframe vs event table
    # ========================================================

    event_ids_df = (
        df.loc[
            mask_stuck,
            "anomalia_id"
        ]
        .dropna()
        .astype(str)
    )

    event_ids_table = (
        events["anomalia_id"]
        .dropna()
        .astype(str)
    )

    unique_ids_df = set(
        event_ids_df.unique()
    )

    unique_ids_table = set(
        event_ids_table.unique()
    )

    ids_only_in_df = (
        unique_ids_df
        - unique_ids_table
    )

    ids_only_in_table = (
        unique_ids_table
        - unique_ids_df
    )

    print(
        f"\nUnique Sensor Atascado IDs in data: "
        f"{len(unique_ids_df):,}"
    )

    print(
        f"Event records: "
        f"{len(events):,}"
    )

    print(
        f"IDs only in dataframe: "
        f"{len(ids_only_in_df):,}"
    )

    print(
        f"IDs only in event table: "
        f"{len(ids_only_in_table):,}"
    )

    if ids_only_in_df:
        print(
            "\nIDs only in dataframe:"
        )
        print(sorted(ids_only_in_df))

    if ids_only_in_table:
        print(
            "\nIDs only in event table:"
        )
        print(sorted(ids_only_in_table))

    if ids_only_in_df or ids_only_in_table:
        traceability_ok = False

    # ========================================================
    # 8. Event count per sensor
    # ========================================================

    print("\nEvents per sensor:")

    events_per_sensor = (
        events["sensor"]
        .value_counts()
        .sort_index()
    )

    for sensor in SENSORES_STUCK:

        actual = int(
            events_per_sensor.get(
                sensor,
                0
            )
        )

        requested = STUCK_EVENTS_PER_SENSOR

        status = (
            "OK"
            if actual == requested
            else "FEWER THAN REQUESTED"
        )

        print(
            f"{sensor:<10}: "
            f"{actual:>2} / {requested:<2} "
            f"({status})"
        )

    # ========================================================
    # 9. Final audit result
    # ========================================================

    print(
        f"\n{dataset_name} traceability: "
        f"{'OK' if traceability_ok else 'FAIL'}"
    )

    if not traceability_ok:

        raise ValueError(
            f"{dataset_name}: Sensor Atascado "
            "traceability audit failed."
        )

    print(
        f"\n✓ {dataset_name} Sensor Atascado audit passed."
    )

    return df_audit


# ============================================================
# 17. RUN AUDITS
# ============================================================

df_audit_stuck_train = audit_sensor_atascado(
    df_train_after_stuck,
    df_events_stuck_train,
    "TRAIN",
)

df_audit_stuck_validation = audit_sensor_atascado(
    df_validation_after_stuck,
    df_events_stuck_validation,
    "VALIDATION",
)

df_audit_stuck_test = audit_sensor_atascado(
    df_test_after_stuck,
    df_events_stuck_test,
    "TEST",
)


# ============================================================
# 18. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("SENSOR ATASCADO — FINAL SUMMARY")
print("=" * 70)

print(
    f"\nTRAIN      events: "
    f"{len(df_events_stuck_train)}"
)

print(
    f"VALIDATION events: "
    f"{len(df_events_stuck_validation)}"
)

print(
    f"TEST       events: "
    f"{len(df_events_stuck_test)}"
)

print(
    f"\nTOTAL events: "
    f"{len(df_events_stuck)}"
)

print("\n✓ Same injection methodology used independently.")
print("✓ No overlap within each dataset.")
print("✓ Original clean datasets remain untouched.")
print("✓ Train / Validation / Test have independent event IDs.")
print("✓ Traceability metadata verified.")

In [ ]:
# Sensors targeted by the professor's anomaly-injection methodology
SENSORES_INYECCION = [
    "PCO2EXT",
    "PHEXT",
    "PRAD",
    "PRGINT",
    "PTEXT",
    "PVV",
    "XCO2I",
    "XHINV",
    "XTINV",
]

# Check that all target sensors exist
missing_sensors = [
    sensor
    for sensor in SENSORES_INYECCION
    if sensor not in df_train_after_stuck.columns
]

if missing_sensors:
    raise ValueError(
        f"Missing injection sensors: {missing_sensors}"
    )

print("Injection sensors:")
print(SENSORES_INYECCION)

In [ ]:
# ============================================================
# DATOS FALTANTES — ANOMALY INJECTION
# ============================================================

def stable_missing_seed(base_seed, dataset_name):
    """
    Create a deterministic dataset-specific seed.
    """
    return stable_seed(
        base_seed,
        dataset_name,
        "missing",
    )


def inject_missing_data(
    df,
    percentage_rows,
    target_sensors,
    min_values=None,
    num_sensors_per_row=1,
    random_state=42,
    used_rows=None,
    dataset_name="DATASET",
):
    """
    Inject missing-data anomalies into one dataset.

    Method adapted from the professor's approach:
    - Only rows without existing NaNs in target sensors
      are candidates.
    - Rows already used by another anomaly are excluded.
    - One target sensor is selected per affected row
      by default.
    - Existing real NaNs are never relabeled.
    - Minimum sensor-value thresholds are respected.
    - Newly affected rows are added to used_rows.

    Traceability:
    - tipo_anomalia
    - sensor_anomalia
    - anomalia_id
    - fuente_anomalia

    For Datos Faltantes:
    - the selected sensor is set to NaN
    - valor_original_anomalia remains NaN
    - valor_inyectado_anomalia remains NaN
      following the existing project convention.
    """

    # ========================================================
    # 1. Basic validation
    # ========================================================

    if not target_sensors:
        print(
            f"{dataset_name}: "
            "No target sensors provided."
        )
        return df.copy(), used_rows

    df_modified = df.copy()

    # ========================================================
    # 2. Global overlap mask
    # ========================================================

    if used_rows is None:

        used_rows = np.zeros(
            len(df_modified),
            dtype=bool,
        )

    if len(used_rows) != len(df_modified):

        raise ValueError(
            f"{dataset_name}: used_rows must have "
            "the same length as the dataframe."
        )

    # ========================================================
    # 3. Keep only sensors that exist
    # ========================================================

    cols = [
        sensor
        for sensor in target_sensors
        if sensor in df_modified.columns
    ]

    if not cols:

        print(
            f"{dataset_name}: None of the target "
            "sensors exist in the dataframe."
        )

        return df_modified, used_rows

    # ========================================================
    # 4. Candidate rows
    # ========================================================

    values = df_modified[
        cols
    ].to_numpy(
        dtype=float
    )

    # No existing NaN in any target sensor
    mask_complete = (
        ~np.isnan(values).any(axis=1)
    )

    # Exclude rows already occupied by
    # previous anomaly types
    mask_candidates = (
        ~used_rows
        & mask_complete
    )

    candidate_indices = np.flatnonzero(
        mask_candidates
    )

    if len(candidate_indices) == 0:

        print(
            f"{dataset_name}: No valid rows "
            "available for missing-data injection."
        )

        return df_modified, used_rows

    # ========================================================
    # 5. Number of rows to affect
    # ========================================================

    n_target = int(
        len(candidate_indices)
        * percentage_rows
    )

    if n_target <= 0:

        print(
            f"{dataset_name}: Target number "
            "of affected rows is 0."
        )

        return df_modified, used_rows

    n_sample = min(
        n_target,
        len(candidate_indices),
    )

    # Dataset-specific deterministic seed
    rng = np.random.default_rng(
        stable_missing_seed(
            random_state,
            dataset_name,
        )
    )

    selected_indices = rng.choice(
        candidate_indices,
        size=n_sample,
        replace=False,
    )

    # ========================================================
    # 6. Determine valid sensor for each row
    # ========================================================

    selected_values = values[
        selected_indices
    ]

    valid_sensor = (
        ~np.isnan(selected_values)
    )

    # Apply minimum-value constraints
    if min_values is not None:

        for j, sensor in enumerate(cols):

            vmin = min_values.get(
                sensor
            )

            if vmin is not None:

                valid_sensor[:, j] &= (
                    selected_values[:, j]
                    >= vmin
                )

    # ========================================================
    # 7. Select one valid sensor per row
    # ========================================================

    valid_rows = []
    selected_sensor_indices = []

    skipped_rows = 0

    for i, row_idx in enumerate(
        selected_indices
    ):

        valid = np.flatnonzero(
            valid_sensor[i]
        )

        if len(valid) == 0:

            skipped_rows += 1
            continue

        if num_sensors_per_row != 1:

            raise NotImplementedError(
                "This implementation currently "
                "supports num_sensors_per_row=1 only."
            )

        chosen = rng.choice(
            valid
        )

        valid_rows.append(
            row_idx
        )

        selected_sensor_indices.append(
            chosen
        )

    if not valid_rows:

        print(
            f"{dataset_name}: No rows satisfied "
            "the sensor-value constraints."
        )

        return df_modified, used_rows

    valid_rows = np.asarray(
        valid_rows,
        dtype=int,
    )

    selected_sensor_indices = np.asarray(
        selected_sensor_indices,
        dtype=int,
    )

    sensors_selected = [
        cols[sensor_idx]
        for sensor_idx in selected_sensor_indices
    ]

    # ========================================================
    # 8. Verify no overlap BEFORE injection
    # ========================================================

    if np.any(
        used_rows[valid_rows]
    ):

        raise RuntimeError(
            f"{dataset_name}: overlap detected "
            "while injecting missing-data anomalies."
        )

    # ========================================================
    # 9. Inject NaN
    # ========================================================

    for row_idx, sensor in zip(
        valid_rows,
        sensors_selected,
    ):

        df_modified.at[
            row_idx,
            sensor
        ] = np.nan

    # ========================================================
    # 10. Mark rows as USED
    # ========================================================

    used_rows[
        valid_rows
    ] = True

    # ========================================================
    # 11. Traceability
    # ========================================================

    if "tipo_anomalia" in df_modified.columns:

        df_modified.loc[
            valid_rows,
            "tipo_anomalia"
        ] = "Datos Faltantes"

    if "sensor_anomalia" in df_modified.columns:

        df_modified.loc[
            valid_rows,
            "sensor_anomalia"
        ] = sensors_selected

    if "anomalia_id" in df_modified.columns:

        for counter, row_idx in enumerate(
            valid_rows,
            start=1,
        ):

            df_modified.at[
                row_idx,
                "anomalia_id"
            ] = (
                f"MF_{dataset_name[:3].upper()}_"
                f"{counter:06d}"
            )

    if "fuente_anomalia" in df_modified.columns:

        df_modified.loc[
            valid_rows,
            "fuente_anomalia"
        ] = "professor_method_missing"

    # --------------------------------------------------------
    # IMPORTANT:
    #
    # Keep both traceability value columns as NaN.
    # This follows the existing project convention for
    # Datos Faltantes.
    # --------------------------------------------------------

    # ========================================================
    # 12. Summary
    # ========================================================

    print(
        f"{dataset_name:<12} | "
        f"Rows modified: {len(valid_rows):,}"
    )

    if skipped_rows:

        print(
            f"{dataset_name:<12} | "
            f"Rows skipped because no valid sensor: "
            f"{skipped_rows:,}"
        )

    print(
        f"{dataset_name:<12} | "
        f"Injected NaN values: {len(valid_rows):,}"
    )

    print(
        f"{dataset_name:<12} | "
        f"Total used rows: {used_rows.sum():,}"
    )

    print(
        f"{dataset_name:<12} | "
        "Affected sensors:"
    )

    sensor_counts = (
        pd.Series(
            sensors_selected
        )
        .value_counts()
        .sort_index()
    )

    print(
        sensor_counts.to_string()
    )

    return (
        df_modified,
        used_rows,
    )

In [ ]:
# ============================================================
# INITIALIZE USED-ROW MASKS FOR MISSING-DATA INJECTION
# ============================================================

used_rows_train = pd.Series(
    False,
    index=df_train_after_stuck.index
)

used_rows_validation = pd.Series(
    False,
    index=df_validation_after_stuck.index
)

used_rows_test = pd.Series(
    False,
    index=df_test_after_stuck.index
)

print("=" * 70)
print("USED-ROW MASKS INITIALIZED")
print("=" * 70)

print(f"TRAIN:      {len(used_rows_train):,} rows")
print(f"VALIDATION: {len(used_rows_validation):,} rows")
print(f"TEST:       {len(used_rows_test):,} rows")

print("\nAlready used:")
print(f"TRAIN:      {used_rows_train.sum():,}")
print(f"VALIDATION: {used_rows_validation.sum():,}")
print(f"TEST:       {used_rows_test.sum():,}")

In [ ]:
# ============================================================
# RUN DATOS FALTANTES — TRAIN / VALIDATION / TEST
# ============================================================

cfg_missing = {
    "percentage_rows": 0.02,
    "num_sensors_per_row": 1,
    "min_values": {
        "PVV": 3.0,
        "PRGINT": 50.0,
    },
}

print("=" * 70)
print("GENERATING DATOS FALTANTES")
print("=" * 70)


# ============================================================
# TRAIN
# ============================================================

df_train_after_missing, used_rows_train = (
    inject_missing_data(
        df=df_train_after_stuck,
        percentage_rows=cfg_missing["percentage_rows"],
        target_sensors=SENSORES_INYECCION,
        min_values=cfg_missing["min_values"],
        num_sensors_per_row=cfg_missing["num_sensors_per_row"],
        random_state=RANDOM_STATE,
        used_rows=used_rows_train,
        dataset_name="TRAIN",
    )
)


# ============================================================
# VALIDATION
# ============================================================

df_validation_after_missing, used_rows_validation = (
    inject_missing_data(
        df=df_validation_after_stuck,
        percentage_rows=cfg_missing["percentage_rows"],
        target_sensors=SENSORES_INYECCION,
        min_values=cfg_missing["min_values"],
        num_sensors_per_row=cfg_missing["num_sensors_per_row"],
        random_state=RANDOM_STATE,
        used_rows=used_rows_validation,
        dataset_name="VALIDATION",
    )
)


# ============================================================
# TEST
# ============================================================

df_test_after_missing, used_rows_test = (
    inject_missing_data(
        df=df_test_after_stuck,
        percentage_rows=cfg_missing["percentage_rows"],
        target_sensors=SENSORES_INYECCION,
        min_values=cfg_missing["min_values"],
        num_sensors_per_row=cfg_missing["num_sensors_per_row"],
        random_state=RANDOM_STATE,
        used_rows=used_rows_test,
        dataset_name="TEST",
    )
)

In [ ]:
# ============================================================
# DATOS FALTANTES — AUDIT
# ============================================================

def audit_missing_data(
    df,
    dataset_name,
):
    """
    Audit Datos Faltantes anomalies for one dataset.
    """

    print("\n" + "=" * 70)
    print(
        f"{dataset_name} — DATOS FALTANTES AUDIT"
    )
    print("=" * 70)

    mask_missing = (
        df["tipo_anomalia"]
        .fillna("")
        .eq("Datos Faltantes")
    )

    n_missing = int(
        mask_missing.sum()
    )

    print(
        f"\nRows Datos Faltantes: "
        f"{n_missing:,}"
    )

    if n_missing == 0:

        print(
            f"\n✓ No Datos Faltantes anomalies "
            f"in {dataset_name}."
        )

        return

    # ========================================================
    # 1. Required traceability
    # ========================================================

    required_columns = [
        "sensor_anomalia",
        "anomalia_id",
        "tipo_anomalia",
        "fuente_anomalia",
    ]

    traceability_ok = True

    print("\nTraceability:")

    for col in required_columns:

        if col not in df.columns:

            print(
                f"{col:<30}: COLUMN MISSING"
            )

            traceability_ok = False
            continue

        missing = int(
            df.loc[
                mask_missing,
                col
            ].isna().sum()
        )

        print(
            f"{col:<30}: "
            f"{missing:,} missing"
        )

        if missing > 0:
            traceability_ok = False

    # ========================================================
    # 2. Correct anomaly type
    # ========================================================

    wrong_type = int(
        df.loc[
            mask_missing,
            "tipo_anomalia"
        ]
        .ne("Datos Faltantes")
        .sum()
    )

    print(
        f"\nWrong anomaly type: "
        f"{wrong_type:,}"
    )

    if wrong_type > 0:
        traceability_ok = False

    # ========================================================
    # 3. Correct source
    # ========================================================

    wrong_source = int(
        df.loc[
            mask_missing,
            "fuente_anomalia"
        ]
        .ne("professor_method_missing")
        .sum()
    )

    print(
        f"Wrong anomaly source: "
        f"{wrong_source:,}"
    )

    if wrong_source > 0:
        traceability_ok = False

    # ========================================================
    # 4. Selected sensor must actually be NaN
    # ========================================================

    nan_injected_ok = True

    print("\nAffected sensors:")

    sensor_counts = (
        df.loc[
            mask_missing,
            "sensor_anomalia"
        ]
        .value_counts()
        .sort_index()
    )

    for sensor, count in sensor_counts.items():

        sensor_mask = (
            mask_missing
            & df["sensor_anomalia"].eq(sensor)
        )

        nan_count = int(
            df.loc[
                sensor_mask,
                sensor
            ].isna().sum()
        )

        total_count = int(
            sensor_mask.sum()
        )

        print(
            f"{sensor:<12}: "
            f"{nan_count:,}/{total_count:,} "
            f"are NaN"
        )

        if nan_count != total_count:
            nan_injected_ok = False

    if not nan_injected_ok:
        traceability_ok = False

    # ========================================================
    # 5. IDs
    # ========================================================

    ids = (
        df.loc[
            mask_missing,
            "anomalia_id"
        ]
        .dropna()
        .astype(str)
    )

    duplicate_ids = int(
        ids.duplicated().sum()
    )

    print(
        f"\nUnique anomaly IDs: "
        f"{ids.nunique():,}"
    )

    print(
        f"Duplicate IDs: "
        f"{duplicate_ids:,}"
    )

    if duplicate_ids > 0:
        traceability_ok = False

    # ========================================================
    # 6. Traceability value columns
    # ========================================================
    #
    # These are intentionally NaN for missing-data anomalies.
    #

    for col in [
        "valor_original_anomalia",
        "valor_inyectado_anomalia",
    ]:

        if col in df.columns:

            non_nan = int(
                df.loc[
                    mask_missing,
                    col
                ].notna().sum()
            )

            print(
                f"{col:<30}: "
                f"{non_nan:,} non-NaN"
            )

            if non_nan > 0:

                print(
                    f"WARNING: {col} contains "
                    "non-NaN values for Datos Faltantes."
                )

    # ========================================================
    # 7. Final result
    # ========================================================

    print(
        f"\n{dataset_name} traceability: "
        f"{'OK' if traceability_ok else 'FAIL'}"
    )

    if not traceability_ok:

        raise ValueError(
            f"{dataset_name}: Datos Faltantes "
            "audit failed."
        )

    print(
        f"\n✓ {dataset_name} Datos Faltantes "
        "audit passed."
    )

In [ ]:
# ============================================================
# RUN AUDITS
# ============================================================

audit_missing_data(
    df_train_after_missing,
    "TRAIN",
)

audit_missing_data(
    df_validation_after_missing,
    "VALIDATION",
)

audit_missing_data(
    df_test_after_missing,
    "TEST",
)

In [ ]:
import sys
PROJECT_DIR = Path("/home/jetson/Desktop/ciesol-anomaly-detection")

if not PROJECT_DIR.exists():
    raise FileNotFoundError(f"Project directory not found: {PROJECT_DIR}")

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# ============================================================
# LOAD PHYSICAL RANGES FROM PROJECT CONFIG
# ============================================================

from src.config_injection import RANGOS_FISICOS

print("RANGOS_FISICOS loaded successfully.")
print(RANGOS_FISICOS)

In [ ]:

# ============================================================
# RUIDO / PICOS — TRAIN ONLY
# ============================================================
#
# Important:
# - Input must be the cumulative dataframe after previous anomalies.
# - used_rows already contains Sensor Atascado + Datos Faltantes.
# - No new used_rows mask is created here.
# - Validation and Test are NOT touched.
# - Injection is performed on physical/unscaled values.
# - Traceability uses the current project metadata columns:
#       sensor_anomalia
#       anomalia_id
#       tipo_anomalia
#       fuente_anomalia
#       valor_original_anomalia
#       valor_inyectado_anomalia
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 1. NOISE CONFIGURATION
# ============================================================

cfg_ruido = {
    "percentage_rows": 0.03,          # 3% of train rows
    "num_sensors_per_row": 1,
    "factor_std_min": 3.0,
    "factor_std_max": 6.0,
}


# ============================================================
# 2. SENSORS USED BY THE PROFESSOR'S NOISE METHOD
# ============================================================

SENSORES_RUIDO = [
    "PCO2EXT",
    "PHEXT",
    "PRAD",
    "PRGINT",
    "PTEXT",
    "PVV",
    "XCO2I",
    "XHINV",
    "XTINV",
]

# Keep only sensors actually present
SENSORES_RUIDO = [
    s
    for s in SENSORES_RUIDO
    if s in df_train_after_missing.columns
]

print("Sensores considerados para Ruido:")
print(SENSORES_RUIDO)


# ============================================================
# 3. SENSOR-SPECIFIC REALISTIC SPIKE MAGNITUDES
# ============================================================

RUIDO_DELTAS = {
    "PTEXT":   {"min": 3.0,   "max": 8.0,   "signos": [-1, 1]},
    "XTINV":   {"min": 3.0,   "max": 8.0,   "signos": [-1, 1]},
    "PHEXT":   {"min": 10.0,  "max": 25.0,  "signos": [-1, 1]},
    "XHINV":   {"min": 10.0,  "max": 25.0,  "signos": [-1, 1]},
    "PCO2EXT": {"min": 25.0,  "max": 75.0,  "signos": [-1, 1]},
    "XCO2I":   {"min": 60.0,  "max": 160.0, "signos": [-1, 1]},
    "PVV":     {"min": 5.0,   "max": 9.0,   "signos": [1]},
    "PRAD":    {"min": 150.0, "max": 500.0, "signos": [1]},
    "PRGINT":  {"min": 80.0,  "max": 250.0, "signos": [1]},
}


# ============================================================
# 4. PHYSICAL RANGES
# ============================================================

# RANGOS_FISICOS is loaded from src.config_injection.
# Do NOT redefine it here.


def _clip_rango_sensor(col, valor):

    if col not in RANGOS_FISICOS:
        return float(valor)

    r = RANGOS_FISICOS[col]

    span = float(
        r["max"] - r["min"]
    )

    eps = max(
        span * 1e-4,
        1e-6
    )

    return float(
        np.clip(
            valor,
            r["min"] + eps,
            r["max"] - eps
        )
    )


# ============================================================
# 5. GENERATE REALISTIC NOISY VALUE
# ============================================================

def _generar_valor_ruido(
    sensor,
    valor_original,
    rng,
):

    cfg = RUIDO_DELTAS[sensor]

    for _ in range(20):

        delta = float(
            rng.uniform(
                cfg["min"],
                cfg["max"]
            )
        )

        signo = float(
            rng.choice(
                cfg["signos"]
            )
        )

        nuevo_valor = _clip_rango_sensor(
            sensor,
            float(valor_original) + signo * delta
        )

        cambio = abs(
            nuevo_valor - float(valor_original)
        )

        # Avoid almost-zero changes caused by clipping
        if cambio >= cfg["min"] * 0.8:

            return (
                nuevo_valor,
                cambio,
                signo,
            )

    return (
        np.nan,
        0.0,
        np.nan,
    )


# ============================================================
# 6. CANDIDATE SELECTION
# ============================================================

# ============================================================
# CANDIDATE SELECTION
# ============================================================

def _candidatos_ruido(
    df,
    sensor,
    used_rows,
):

    valores = pd.to_numeric(
        df[sensor],
        errors="coerce"
    )

    fechas = pd.to_datetime(
        df["FECHA"],
        errors="coerce"
    )

    mask = (
        valores.notna()
        & fechas.notna()
    )

    # --------------------------------------------------------
    # Physical range
    # --------------------------------------------------------

    if sensor in RANGOS_FISICOS:

        r = RANGOS_FISICOS[sensor]

        mask &= valores.between(
            r["min"],
            r["max"]
        )

    # --------------------------------------------------------
    # Sensor-specific restrictions
    # --------------------------------------------------------

    if sensor == "PRAD":

        # Radiation spike only during daylight
        mask &= (
            (valores >= 50.0)
            & fechas.dt.hour.between(8, 19)
        )

    elif sensor == "PRGINT":

        mask &= valores >= 100.0

    elif sensor == "PVV":

        mask &= valores >= 5.0

    elif sensor == "PCO2EXT":

        if "RUIDO_VALOR_MINIMO_INYECCION" in globals():

            minimo = RUIDO_VALOR_MINIMO_INYECCION.get(
                "PCO2EXT",
                -np.inf
            )

            mask &= valores >= minimo

    # --------------------------------------------------------
    # Exclude rows already used by previous anomalies
    # --------------------------------------------------------

    used_rows_array = np.asarray(
        used_rows,
        dtype=bool
    )

    if len(used_rows_array) != len(df):

        raise ValueError(
            "used_rows must have the same length as df."
        )

    mask &= ~used_rows_array

    # --------------------------------------------------------
    # Return POSITIONAL indices
    # --------------------------------------------------------

    return np.flatnonzero(
        mask.to_numpy()
    )


# ============================================================
# 7. MAIN RUIDO INJECTION
# ============================================================

# ============================================================
# MAIN RUIDO INJECTION
# ============================================================

def inyectar_ruido(
    df,
    porcentaje_filas_ruido,
    columnas_objetivo,
    random_state=42,
    used_rows=None,
    dataset_name="DATASET",
):

    df_modificado = df.copy()

    # --------------------------------------------------------
    # Global overlap mask
    # --------------------------------------------------------

    if used_rows is None:

        used_rows = np.zeros(
            len(df_modificado),
            dtype=bool
        )

    else:

        used_rows = np.asarray(
            used_rows,
            dtype=bool
        ).copy()

        if len(used_rows) != len(df_modificado):

            raise ValueError(
                "used_rows must have the same length as df."
            )

    # --------------------------------------------------------
    # Required traceability columns
    # --------------------------------------------------------

    required_columns = [
        "sensor_anomalia",
        "anomalia_id",
        "tipo_anomalia",
        "fuente_anomalia",
        "valor_original_anomalia",
        "valor_inyectado_anomalia",
    ]

    missing_columns = [
        c for c in required_columns
        if c not in df_modificado.columns
    ]

    if missing_columns:

        raise ValueError(
            f"Missing traceability columns: "
            f"{missing_columns}"
        )

    # --------------------------------------------------------
    # Valid sensors
    # --------------------------------------------------------

    sensores = [
        s
        for s in columnas_objetivo
        if s in RUIDO_DELTAS
        and s in df_modificado.columns
    ]

    if not sensores:

        print(
            f"{dataset_name}: "
            "No hay sensores válidos para Ruido."
        )

        return (
            df_modificado,
            used_rows,
            pd.DataFrame(),
        )

    # --------------------------------------------------------
    # Number of rows
    # --------------------------------------------------------

    n_rows = len(df_modificado)

    objetivo_total = int(
        round(
            n_rows
            * porcentaje_filas_ruido
        )
    )

    print(
        f"\n{dataset_name} | "
        f"Objetivo total de filas con Ruido: "
        f"{objetivo_total:,}"
    )

    # --------------------------------------------------------
    # Approximately equal distribution
    # --------------------------------------------------------

    base_por_sensor = (
        objetivo_total
        // len(sensores)
    )

    resto = (
        objetivo_total
        % len(sensores)
    )

    cuotas = {}

    for i, sensor in enumerate(sensores):

        cuotas[sensor] = (
            base_por_sensor
            + (
                1
                if i < resto
                else 0
            )
        )

    print(
        f"\n{dataset_name} | "
        "Cuotas de Ruido por sensor:"
    )

    for sensor, cuota in cuotas.items():

        print(
            f"  {sensor:8s}: "
            f"{cuota:,}"
        )

    # --------------------------------------------------------
    # Column positions
    # --------------------------------------------------------

    col_indices = {
        sensor:
        df_modificado.columns.get_loc(sensor)
        for sensor in sensores
    }

    idx_tipo = df_modificado.columns.get_loc(
        "tipo_anomalia"
    )

    idx_sensor = df_modificado.columns.get_loc(
        "sensor_anomalia"
    )

    idx_id = df_modificado.columns.get_loc(
        "anomalia_id"
    )

    idx_fuente = df_modificado.columns.get_loc(
        "fuente_anomalia"
    )

    idx_original = df_modificado.columns.get_loc(
        "valor_original_anomalia"
    )

    idx_inyectado = df_modificado.columns.get_loc(
        "valor_inyectado_anomalia"
    )

    registros = []

    # --------------------------------------------------------
    # Injection by sensor
    # --------------------------------------------------------

    for sensor in sensores:

        objetivo = cuotas[sensor]

        if objetivo <= 0:
            continue

        candidatos = _candidatos_ruido(
            df_modificado,
            sensor,
            used_rows,
        )

        rng = np.random.default_rng(
            stable_seed(
                random_state,
                "ruido",
                dataset_name,
                sensor,
            )
        )

        candidatos = rng.permutation(
            candidatos
        )

        creadas = 0
        rechazadas = 0

        for fila_idx in candidatos:

            if creadas >= objetivo:
                break

            # ------------------------------------------------
            # Never overlap previous anomaly
            # ------------------------------------------------

            if used_rows[fila_idx]:
                continue

            valor_original = (
                df_modificado.iat[
                    fila_idx,
                    col_indices[sensor]
                ]
            )

            if pd.isna(valor_original):
                continue

            nuevo_valor, cambio, signo = (
                _generar_valor_ruido(
                    sensor,
                    float(valor_original),
                    rng,
                )
            )

            if (
                not np.isfinite(nuevo_valor)
                or np.isclose(
                    nuevo_valor,
                    float(valor_original),
                    rtol=0.0,
                    atol=1e-12,
                )
            ):

                rechazadas += 1
                continue

            # ------------------------------------------------
            # Final overlap safety check
            # ------------------------------------------------

            if used_rows[fila_idx]:

                raise RuntimeError(
                    f"{dataset_name}: "
                    "Overlap detected during "
                    "Ruido injection."
                )

            # ------------------------------------------------
            # Inject value
            # ------------------------------------------------

            df_modificado.iat[
                fila_idx,
                col_indices[sensor]
            ] = float(nuevo_valor)

            # ------------------------------------------------
            # Dataset-specific ID
            # ------------------------------------------------

            anomaly_id = (
                f"RUIDO_{dataset_name}_"
                f"{sensor}_{fila_idx:06d}"
            )

            # ------------------------------------------------
            # Traceability
            # ------------------------------------------------

            df_modificado.iat[
                fila_idx,
                idx_tipo
            ] = "Ruido"

            df_modificado.iat[
                fila_idx,
                idx_sensor
            ] = sensor

            df_modificado.iat[
                fila_idx,
                idx_id
            ] = anomaly_id

            df_modificado.iat[
                fila_idx,
                idx_fuente
            ] = "professor_method_noise"

            df_modificado.iat[
                fila_idx,
                idx_original
            ] = float(valor_original)

            df_modificado.iat[
                fila_idx,
                idx_inyectado
            ] = float(nuevo_valor)

            # ------------------------------------------------
            # Mark row as used
            # ------------------------------------------------

            used_rows[fila_idx] = True

            creadas += 1

            # ------------------------------------------------
            # Event record
            # ------------------------------------------------

            registros.append({

                "dataset":
                    dataset_name,

                "tipo_anomalia":
                    "Ruido",

                "sensor_anomalia":
                    sensor,

                "anomalia_id":
                    anomaly_id,

                "fuente_anomalia":
                    "professor_method_noise",

                "idx":
                    int(fila_idx),

                "FECHA":
                    df_modificado.iloc[
                        fila_idx
                    ]["FECHA"],

                "valor_original_anomalia":
                    float(valor_original),

                "valor_inyectado_anomalia":
                    float(nuevo_valor),

                "delta":
                    float(
                        nuevo_valor
                        - float(valor_original)
                    ),

                "abs_delta":
                    float(cambio),

                "signo":
                    float(signo),

                "rng_seed":
                    int(
                        stable_seed(
                            random_state,
                            "ruido",
                            dataset_name,
                            sensor,
                        )
                    ),
            })

        print(
            f"{dataset_name} | "
            f"Ruido {sensor:8s}: "
            f"{creadas:,}/{objetivo:,} "
            f"(candidatos={len(candidatos):,}, "
            f"rechazadas={rechazadas:,})"
        )

    # --------------------------------------------------------
    # Event dataframe
    # --------------------------------------------------------

    df_eventos_ruido = pd.DataFrame(
        registros
    )

    print(
        f"\n{dataset_name} | "
        f"Inyección de Ruido completada: "
        f"{len(df_eventos_ruido):,} filas modificadas."
    )

    print(
        f"{dataset_name} | "
        f"Filas ocupadas por anomalías acumuladas: "
        f"{int(used_rows.sum()):,}"
    )

    return (
        df_modificado,
        used_rows,
        df_eventos_ruido,
    )



In [ ]:
df_train_after_ruido, used_rows_train, df_eventos_ruido_train = (
    inyectar_ruido(
        df=df_train_after_missing,
        porcentaje_filas_ruido=cfg_ruido["percentage_rows"],
        columnas_objetivo=SENSORES_RUIDO,
        random_state=RANDOM_STATE,
        used_rows=used_rows_train,
        dataset_name="TRAIN",
    )
)

In [ ]:
df_validation_after_ruido, used_rows_validation, df_eventos_ruido_validation = (
    inyectar_ruido(
        df=df_validation_after_missing,
        porcentaje_filas_ruido=cfg_ruido["percentage_rows"],
        columnas_objetivo=SENSORES_RUIDO,
        random_state=RANDOM_STATE,
        used_rows=used_rows_validation,
        dataset_name="VALIDATION",
    )
)

In [ ]:
df_test_after_ruido, used_rows_test, df_eventos_ruido_test = (
    inyectar_ruido(
        df=df_test_after_missing,
        porcentaje_filas_ruido=cfg_ruido["percentage_rows"],
        columnas_objetivo=SENSORES_RUIDO,
        random_state=RANDOM_STATE,
        used_rows=used_rows_test,
        dataset_name="TEST",
    )
)

In [ ]:
# ============================================================
# RUIDO — AUDIT
# ============================================================

def audit_ruido(
    df,
    df_eventos_ruido,
    used_rows,
    dataset_name,
):

    print("\n" + "=" * 70)
    print(f"{dataset_name} — RUIDO / PICOS AUDIT")
    print("=" * 70)

    mask_ruido = (
        df["tipo_anomalia"]
        .eq("Ruido")
    )

    n_ruido = int(mask_ruido.sum())

    print(
        f"\nRows Ruido: {n_ruido:,}"
    )

    if n_ruido == 0:

        print(
            f"WARNING: {dataset_name} "
            "contains no Ruido rows."
        )

        return False

    # --------------------------------------------------------
    # Metadata
    # --------------------------------------------------------

    checks = {

        "sensor_anomalia":
            df.loc[
                mask_ruido,
                "sensor_anomalia"
            ].notna().all(),

        "anomalia_id":
            df.loc[
                mask_ruido,
                "anomalia_id"
            ].notna().all(),

        "tipo_anomalia":
            df.loc[
                mask_ruido,
                "tipo_anomalia"
            ].notna().all(),

        "fuente_anomalia":
            df.loc[
                mask_ruido,
                "fuente_anomalia"
            ].notna().all(),

        "valor_original_anomalia":
            df.loc[
                mask_ruido,
                "valor_original_anomalia"
            ].notna().all(),

        "valor_inyectado_anomalia":
            df.loc[
                mask_ruido,
                "valor_inyectado_anomalia"
            ].notna().all(),
    }

    print("\nTraceability:")

    for col, ok in checks.items():

        missing = int(
            df.loc[
                mask_ruido,
                col
            ].isna().sum()
        )

        print(
            f"{col:30s}: "
            f"{missing} missing"
        )

    # --------------------------------------------------------
    # Correct type/source
    # --------------------------------------------------------

    wrong_type = int(
        (
            df.loc[
                mask_ruido,
                "tipo_anomalia"
            ]
            != "Ruido"
        ).sum()
    )

    wrong_source = int(
        (
            df.loc[
                mask_ruido,
                "fuente_anomalia"
            ]
            != "professor_method_noise"
        ).sum()
    )

    print(
        f"\nWrong anomaly type: {wrong_type}"
    )

    print(
        f"Wrong anomaly source: {wrong_source}"
    )

    # --------------------------------------------------------
    # Sensor counts
    # --------------------------------------------------------

    print("\nAffected sensors:")

    sensor_counts = (
        df.loc[
            mask_ruido,
            "sensor_anomalia"
        ]
        .value_counts()
        .sort_index()
    )

    print(sensor_counts)

    # --------------------------------------------------------
    # IDs
    # --------------------------------------------------------

    ids = (
        df.loc[
            mask_ruido,
            "anomalia_id"
        ]
    )

    duplicate_ids = int(
        ids.duplicated().sum()
    )

    print(
        f"\nUnique anomaly IDs: "
        f"{ids.nunique():,}"
    )

    print(
        f"Duplicate IDs: "
        f"{duplicate_ids}"
    )

    # --------------------------------------------------------
    # Verify original != injected
    # --------------------------------------------------------

    original = pd.to_numeric(
        df.loc[
            mask_ruido,
            "valor_original_anomalia"
        ],
        errors="coerce"
    )

    injected = pd.to_numeric(
        df.loc[
            mask_ruido,
            "valor_inyectado_anomalia"
        ],
        errors="coerce"
    )

    same_value = int(
        np.isclose(
            original.to_numpy(),
            injected.to_numpy(),
            rtol=0.0,
            atol=1e-12,
        ).sum()
    )

    print(
        f"\nOriginal == injected: "
        f"{same_value}"
    )

    # --------------------------------------------------------
    # Used rows
    # --------------------------------------------------------

    ruido_indices = (
        df_eventos_ruido["idx"]
        .astype(int)
        .to_numpy()
    )

    used_ok = (
        len(ruido_indices) == 0
        or np.all(
            np.asarray(
                used_rows,
                dtype=bool
            )[ruido_indices]
        )
    )

    print(
        f"Ruido rows marked in used_rows: "
        f"{used_ok}"
    )

    # --------------------------------------------------------
    # Final pass
    # --------------------------------------------------------

    passed = all(checks.values())

    passed &= (
        wrong_type == 0
        and wrong_source == 0
        and duplicate_ids == 0
        and same_value == 0
        and used_ok
    )

    if passed:

        print(
            f"\n✓ {dataset_name} "
            "Ruido audit passed."
        )

    else:

        print(
            f"\n✗ {dataset_name} "
            "Ruido audit FAILED."
        )

    return passed

In [ ]:
audit_ruido(
    df_train_after_ruido,
    df_eventos_ruido_train,
    used_rows_train,
    "TRAIN",
)

audit_ruido(
    df_validation_after_ruido,
    df_eventos_ruido_validation,
    used_rows_validation,
    "VALIDATION",
)

audit_ruido(
    df_test_after_ruido,
    df_eventos_ruido_test,
    used_rows_test,
    "TEST",
)

In [ ]:

# ============================================================
# VALORES FUERA DE RANGO — TRAIN ONLY
# ============================================================
#
# Pipeline:
#
#   Sensor Atascado
#          ↓
#   Datos Faltantes
#          ↓
#   Ruido
#          ↓
#   Valores Fuera de Rango
#
# IMPORTANT:
# - df_train_after_ruido is the cumulative Train dataframe.
# - used_rows already contains all previous anomaly rows.
# - DO NOT recreate used_rows here.
# - Validation and Test are NOT modified.
# - Injection is performed before scaling.
# - Traceability uses the current project metadata columns.
# ============================================================


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

cfg_oor = {
    "percentage_rows": 0.02,       # 2% of Train rows
    "num_sensors_per_row": 1,      # one sensor per row
    "margin_error_factor": 0.10,   # 10% outside physical range
}


# ------------------------------------------------------------
# 2. Sensors
# ------------------------------------------------------------

SENSORES_OOR = [
    "PCO2EXT",
    "PHEXT",
    "PRAD",
    "PRGINT",
    "PTEXT",
    "PVV",
    "XCO2I",
    "XHINV",
    "XTINV",
]

SENSORES_OOR = [
    sensor
    for sensor in SENSORES_OOR
    if sensor in df_train_after_ruido.columns
]

print(
    "Sensores considerados para "
    "Valores Fuera de Rango:"
)

print(SENSORES_OOR)


# ------------------------------------------------------------
# 3. Physical ranges
# ------------------------------------------------------------
#
# RANGOS_FISICOS should already be loaded from
# config_injection.py.
#
# If it is already defined, use it directly.
# ------------------------------------------------------------

if "RANGOS_FISICOS" not in globals():

    RANGOS_FISICOS = {
        "PCO2EXT": {"min": 300.0, "max": 800.0},
        "PHEXT":   {"min": 0.0,   "max": 100.0},
        "PRAD":    {"min": 0.0,   "max": 1500.0},
        "PRGINT":  {"min": 0.0,   "max": 1200.0},
        "PTEXT":   {"min": -20.0, "max": 60.0},
        "PVV":     {"min": 0.0,   "max": 50.0},
        "XCO2I":   {"min": 200.0, "max": 3000.0},
        "XHINV":   {"min": 0.0,   "max": 100.0},
        "XTINV":   {"min": -5.0,  "max": 60.0},
    }


print("\nRangos físicos utilizados:")

for sensor in SENSORES_OOR:

    r = RANGOS_FISICOS[sensor]

    print(
        f"  {sensor:8s}: "
        f"{r['min']:8.1f} -> "
        f"{r['max']:8.1f}"
    )


# ------------------------------------------------------------
# 4. Generate out-of-range value
# ------------------------------------------------------------

def _generar_valor_fuera_rango(
    sensor,
    valor_original,
    rng,
    margen_error_factor=0.10,
):

    r = RANGOS_FISICOS[sensor]

    sensor_min = float(
        r["min"]
    )

    sensor_max = float(
        r["max"]
    )

    span = (
        sensor_max
        - sensor_min
    )

    # 10% of physical range
    margen = (
        span
        * margen_error_factor
    )

    # Randomly choose below minimum
    # or above maximum
    if rng.random() < 0.5:

        nuevo_valor = (
            sensor_min
            - margen
        )

        direccion = "below_min"

    else:

        nuevo_valor = (
            sensor_max
            + margen
        )

        direccion = "above_max"

    # Make sure the injected value is genuinely different
    if np.isclose(
        nuevo_valor,
        float(valor_original),
        rtol=0.0,
        atol=1e-12,
    ):

        return (
            np.nan,
            direccion,
        )

    return (
        float(nuevo_valor),
        direccion,
    )


# ------------------------------------------------------------
# 5. Candidate selection
# ------------------------------------------------------------

# ============================================================
# CANDIDATE SELECTION
# ============================================================

def _candidatos_oor(
    df,
    sensor,
    used_rows,
):

    valores = pd.to_numeric(
        df[sensor],
        errors="coerce"
    )

    r = RANGOS_FISICOS[sensor]

    mask = (
        valores.notna()
        & valores.between(
            r["min"],
            r["max"]
        )
    )

    # --------------------------------------------------------
    # Exclude all rows already used by previous anomalies
    # --------------------------------------------------------

    used_rows = np.asarray(
        used_rows,
        dtype=bool
    )

    if len(used_rows) != len(df):

        raise ValueError(
            "used_rows must have the same "
            "length as df."
        )

    mask &= ~used_rows

    # --------------------------------------------------------
    # Return positional indices
    # --------------------------------------------------------

    return np.flatnonzero(
        mask.to_numpy()
    )


# ------------------------------------------------------------
# 6. Main OOR injection
# ------------------------------------------------------------

# ============================================================
# MAIN OOR INJECTION
# ============================================================

def inyectar_valores_fuera_rango(
    df,
    porcentaje_filas_afectadas,
    columnas_objetivo,
    num_sensores_por_fila=1,
    margen_error_factor=0.10,
    random_state=42,
    used_rows=None,
    dataset_name="DATASET",
):

    df_modificado = df.copy()

    # --------------------------------------------------------
    # Global cumulative used-row mask
    # --------------------------------------------------------

    if used_rows is None:

        used_rows = np.zeros(
            len(df_modificado),
            dtype=bool
        )

    else:

        used_rows = np.asarray(
            used_rows,
            dtype=bool
        ).copy()

        if len(used_rows) != len(df_modificado):

            raise ValueError(
                "used_rows must have the same "
                "length as df."
            )

    # --------------------------------------------------------
    # Required traceability columns
    # --------------------------------------------------------

    required_columns = [
        "sensor_anomalia",
        "anomalia_id",
        "tipo_anomalia",
        "fuente_anomalia",
        "valor_original_anomalia",
        "valor_inyectado_anomalia",
    ]

    missing_columns = [
        col
        for col in required_columns
        if col not in df_modificado.columns
    ]

    if missing_columns:

        raise ValueError(
            f"Missing traceability columns: "
            f"{missing_columns}"
        )

    # --------------------------------------------------------
    # Valid sensors
    # --------------------------------------------------------

    sensores = [
        sensor
        for sensor in columnas_objetivo
        if sensor in RANGOS_FISICOS
        and sensor in df_modificado.columns
    ]

    if not sensores:

        print(
            f"{dataset_name}: "
            "No hay sensores válidos para OOR."
        )

        return (
            df_modificado,
            used_rows,
            pd.DataFrame(),
        )

    # --------------------------------------------------------
    # Total target
    # --------------------------------------------------------

    n_rows = len(df_modificado)

    objetivo_total = int(
        round(
            n_rows
            * porcentaje_filas_afectadas
        )
    )

    print(
        f"\n{dataset_name} | "
        f"Objetivo total de filas OOR: "
        f"{objetivo_total:,}"
    )

    # --------------------------------------------------------
    # Distribute approximately equally
    # --------------------------------------------------------

    base_por_sensor = (
        objetivo_total
        // len(sensores)
    )

    resto = (
        objetivo_total
        % len(sensores)
    )

    cuotas = {}

    for i, sensor in enumerate(sensores):

        cuotas[sensor] = (
            base_por_sensor
            + (
                1
                if i < resto
                else 0
            )
        )

    print(
        f"\n{dataset_name} | "
        "Cuotas OOR por sensor:"
    )

    for sensor, cuota in cuotas.items():

        print(
            f"  {sensor:8s}: "
            f"{cuota:,}"
        )

    # --------------------------------------------------------
    # Column positions
    # --------------------------------------------------------

    col_indices = {
        sensor:
        df_modificado.columns.get_loc(sensor)
        for sensor in sensores
    }

    idx_tipo = df_modificado.columns.get_loc(
        "tipo_anomalia"
    )

    idx_sensor = df_modificado.columns.get_loc(
        "sensor_anomalia"
    )

    idx_id = df_modificado.columns.get_loc(
        "anomalia_id"
    )

    idx_fuente = df_modificado.columns.get_loc(
        "fuente_anomalia"
    )

    idx_original = df_modificado.columns.get_loc(
        "valor_original_anomalia"
    )

    idx_inyectado = df_modificado.columns.get_loc(
        "valor_inyectado_anomalia"
    )

    registros = []

    # --------------------------------------------------------
    # Injection
    # --------------------------------------------------------

    for sensor in sensores:

        objetivo = cuotas[sensor]

        if objetivo <= 0:
            continue

        candidatos = _candidatos_oor(
            df_modificado,
            sensor,
            used_rows,
        )

        rng = np.random.default_rng(
            stable_seed(
                random_state,
                "fuera_rango",
                dataset_name,
                sensor,
            )
        )

        candidatos = rng.permutation(
            candidatos
        )

        creadas = 0
        rechazadas = 0

        for fila_idx in candidatos:

            if creadas >= objetivo:
                break

            # ------------------------------------------------
            # Never overlap previous anomalies
            # ------------------------------------------------

            if used_rows[fila_idx]:
                continue

            col_idx = col_indices[sensor]

            valor_original = (
                df_modificado.iat[
                    fila_idx,
                    col_idx
                ]
            )

            if pd.isna(valor_original):
                continue

            # ------------------------------------------------
            # Generate OOR value
            # ------------------------------------------------

            nuevo_valor, direccion = (
                _generar_valor_fuera_rango(
                    sensor,
                    float(valor_original),
                    rng,
                    margen_error_factor,
                )
            )

            if not np.isfinite(nuevo_valor):

                rechazadas += 1
                continue

            # ------------------------------------------------
            # Safety check
            # ------------------------------------------------

            if used_rows[fila_idx]:

                raise RuntimeError(
                    f"{dataset_name}: "
                    "Overlap detected during "
                    "OOR injection."
                )

            # ------------------------------------------------
            # Inject
            # ------------------------------------------------

            df_modificado.iat[
                fila_idx,
                col_idx
            ] = float(nuevo_valor)

            # ------------------------------------------------
            # Dataset-specific anomaly ID
            # ------------------------------------------------

            anomaly_id = (
                f"OOR_{dataset_name}_"
                f"{sensor}_{fila_idx:06d}"
            )

            # ------------------------------------------------
            # Traceability
            # ------------------------------------------------

            df_modificado.iat[
                fila_idx,
                idx_tipo
            ] = "Valores Fuera de Rango"

            df_modificado.iat[
                fila_idx,
                idx_sensor
            ] = sensor

            df_modificado.iat[
                fila_idx,
                idx_id
            ] = anomaly_id

            df_modificado.iat[
                fila_idx,
                idx_fuente
            ] = (
                "professor_method_out_of_range"
            )

            df_modificado.iat[
                fila_idx,
                idx_original
            ] = float(valor_original)

            df_modificado.iat[
                fila_idx,
                idx_inyectado
            ] = float(nuevo_valor)

            # ------------------------------------------------
            # Mark row as used
            # ------------------------------------------------

            used_rows[fila_idx] = True

            creadas += 1

            # ------------------------------------------------
            # Event record
            # ------------------------------------------------

            registros.append({

                "dataset":
                    dataset_name,

                "tipo_anomalia":
                    "Valores Fuera de Rango",

                "sensor_anomalia":
                    sensor,

                "anomalia_id":
                    anomaly_id,

                "fuente_anomalia":
                    "professor_method_out_of_range",

                "idx":
                    int(fila_idx),

                "FECHA":
                    df_modificado.iloc[
                        fila_idx
                    ]["FECHA"],

                "valor_original_anomalia":
                    float(valor_original),

                "valor_inyectado_anomalia":
                    float(nuevo_valor),

                "direccion":
                    direccion,

                "limite_min":
                    float(
                        RANGOS_FISICOS[
                            sensor
                        ]["min"]
                    ),

                "limite_max":
                    float(
                        RANGOS_FISICOS[
                            sensor
                        ]["max"]
                    ),

                "margen_error_factor":
                    float(
                        margen_error_factor
                    ),

                "rng_seed":
                    int(
                        stable_seed(
                            random_state,
                            "fuera_rango",
                            dataset_name,
                            sensor,
                        )
                    ),
            })

        estado = (
            "OK"
            if creadas == objetivo
            else
            f"AVISO {creadas}/{objetivo}"
        )

        print(
            f"{dataset_name} | "
            f"OOR {sensor:8s}: "
            f"{creadas:,}/{objetivo:,} "
            f"(candidatos={len(candidatos):,}, "
            f"rechazadas={rechazadas:,}) "
            f"{estado}"
        )

    # --------------------------------------------------------
    # Event dataframe
    # --------------------------------------------------------

    df_eventos_oor = pd.DataFrame(
        registros
    )

    print(
        f"\n{dataset_name} | "
        f"Inyección OOR completada: "
        f"{len(df_eventos_oor):,} "
        f"filas modificadas."
    )

    print(
        f"{dataset_name} | "
        f"Filas ocupadas acumuladas: "
        f"{int(used_rows.sum()):,}"
    )

    return (
        df_modificado,
        used_rows,
        df_eventos_oor,
    )


In [ ]:
# ============================================================
# RUN OOR — TRAIN
# ============================================================

df_train_after_oor, used_rows_train, df_eventos_oor_train = (
    inyectar_valores_fuera_rango(
        df=df_train_after_ruido,
        porcentaje_filas_afectadas=
            cfg_oor["percentage_rows"],
        columnas_objetivo=
            SENSORES_OOR,
        num_sensores_por_fila=
            cfg_oor["num_sensors_per_row"],
        margen_error_factor=
            cfg_oor["margin_error_factor"],
        random_state=
            RANDOM_STATE,
        used_rows=
            used_rows_train,
        dataset_name="TRAIN",
    )
)


# ============================================================
# RUN OOR — VALIDATION
# ============================================================

df_validation_after_oor, used_rows_validation, df_eventos_oor_validation = (
    inyectar_valores_fuera_rango(
        df=df_validation_after_ruido,
        porcentaje_filas_afectadas=
            cfg_oor["percentage_rows"],
        columnas_objetivo=
            SENSORES_OOR,
        num_sensores_por_fila=
            cfg_oor["num_sensors_per_row"],
        margen_error_factor=
            cfg_oor["margin_error_factor"],
        random_state=
            RANDOM_STATE,
        used_rows=
            used_rows_validation,
        dataset_name="VALIDATION",
    )
)


# ============================================================
# RUN OOR — TEST
# ============================================================

df_test_after_oor, used_rows_test, df_eventos_oor_test = (
    inyectar_valores_fuera_rango(
        df=df_test_after_ruido,
        porcentaje_filas_afectadas=
            cfg_oor["percentage_rows"],
        columnas_objetivo=
            SENSORES_OOR,
        num_sensores_por_fila=
            cfg_oor["num_sensors_per_row"],
        margen_error_factor=
            cfg_oor["margin_error_factor"],
        random_state=
            RANDOM_STATE,
        used_rows=
            used_rows_test,
        dataset_name="TEST",
    )
)

In [ ]:
# ============================================================
# OOR — AUDIT
# ============================================================

def audit_oor(
    df,
    df_eventos_oor,
    used_rows,
    dataset_name,
):

    print("\n" + "=" * 70)
    print(
        f"{dataset_name} — "
        "VALORES FUERA DE RANGO AUDIT"
    )
    print("=" * 70)

    mask_oor = (
        df["tipo_anomalia"]
        .eq("Valores Fuera de Rango")
    )

    n_oor = int(mask_oor.sum())

    print(
        f"\nRows OOR: {n_oor:,}"
    )

    if n_oor == 0:

        print(
            f"WARNING: {dataset_name} "
            "contains no OOR rows."
        )

        return False

    # --------------------------------------------------------
    # Traceability
    # --------------------------------------------------------

    columns_to_check = [
        "sensor_anomalia",
        "anomalia_id",
        "tipo_anomalia",
        "fuente_anomalia",
        "valor_original_anomalia",
        "valor_inyectado_anomalia",
    ]

    print("\nTraceability:")

    traceability_ok = True

    for col in columns_to_check:

        missing = int(
            df.loc[
                mask_oor,
                col
            ].isna().sum()
        )

        print(
            f"{col:30s}: "
            f"{missing} missing"
        )

        if missing != 0:
            traceability_ok = False

    # --------------------------------------------------------
    # Type / source
    # --------------------------------------------------------

    wrong_type = int(
        (
            df.loc[
                mask_oor,
                "tipo_anomalia"
            ]
            != "Valores Fuera de Rango"
        ).sum()
    )

    wrong_source = int(
        (
            df.loc[
                mask_oor,
                "fuente_anomalia"
            ]
            != "professor_method_out_of_range"
        ).sum()
    )

    print(
        f"\nWrong anomaly type: {wrong_type}"
    )

    print(
        f"Wrong anomaly source: {wrong_source}"
    )

    # --------------------------------------------------------
    # Sensors
    # --------------------------------------------------------

    print("\nAffected sensors:")

    print(
        df.loc[
            mask_oor,
            "sensor_anomalia"
        ]
        .value_counts()
        .sort_index()
    )

    # --------------------------------------------------------
    # IDs
    # --------------------------------------------------------

    ids = df.loc[
        mask_oor,
        "anomalia_id"
    ]

    duplicate_ids = int(
        ids.duplicated().sum()
    )

    print(
        f"\nUnique anomaly IDs: "
        f"{ids.nunique():,}"
    )

    print(
        f"Duplicate IDs: "
        f"{duplicate_ids}"
    )

    # --------------------------------------------------------
    # Original != injected
    # --------------------------------------------------------

    original = pd.to_numeric(
        df.loc[
            mask_oor,
            "valor_original_anomalia"
        ],
        errors="coerce"
    )

    injected = pd.to_numeric(
        df.loc[
            mask_oor,
            "valor_inyectado_anomalia"
        ],
        errors="coerce"
    )

    same_value = int(
        np.isclose(
            original.to_numpy(),
            injected.to_numpy(),
            rtol=0.0,
            atol=1e-12,
        ).sum()
    )

    print(
        f"\nOriginal == injected: "
        f"{same_value}"
    )

    # --------------------------------------------------------
    # Verify physical range
    # --------------------------------------------------------

    errors_outside = 0

    for _, row in df.loc[
        mask_oor,
        [
            "sensor_anomalia",
            "valor_inyectado_anomalia",
        ]
    ].iterrows():

        sensor = row[
            "sensor_anomalia"
        ]

        value = row[
            "valor_inyectado_anomalia"
        ]

        r = RANGOS_FISICOS[sensor]

        if not (
            value < r["min"]
            or value > r["max"]
        ):

            errors_outside += 1

    print(
        f"Injected values NOT outside range: "
        f"{errors_outside}"
    )

    # --------------------------------------------------------
    # Verify used_rows
    # --------------------------------------------------------

    oor_indices = (
        df_eventos_oor["idx"]
        .astype(int)
        .to_numpy()
    )

    used_rows_array = np.asarray(
        used_rows,
        dtype=bool
    )

    used_ok = (
        len(oor_indices) == 0
        or np.all(
            used_rows_array[
                oor_indices
            ]
        )
    )

    print(
        f"All OOR rows marked in used_rows: "
        f"{used_ok}"
    )

    # --------------------------------------------------------
    # Final result
    # --------------------------------------------------------

    passed = (
        traceability_ok
        and wrong_type == 0
        and wrong_source == 0
        and duplicate_ids == 0
        and same_value == 0
        and errors_outside == 0
        and used_ok
    )

    if passed:

        print(
            f"\n✓ {dataset_name} "
            "OOR audit passed."
        )

    else:

        print(
            f"\n✗ {dataset_name} "
            "OOR audit FAILED."
        )

    return passed

In [ ]:
audit_oor(
    df_train_after_oor,
    df_eventos_oor_train,
    used_rows_train,
    "TRAIN",
)

audit_oor(
    df_validation_after_oor,
    df_eventos_oor_validation,
    used_rows_validation,
    "VALIDATION",
)

audit_oor(
    df_test_after_oor,
    df_eventos_oor_test,
    used_rows_test,
    "TEST",
)

In [ ]:

# Ventilation / actuator columns
COLUMNAS_VENTILACION = [
    "UVENT_cen",
    "UVENT_lN",
]

print("Columnas de ventilación:", COLUMNAS_VENTILACION)



In [ ]:
# ============================================================
# DESVIACIÓN DE CORRELACIÓN — TRAIN / VALIDATION / TEST
# ============================================================
#
# Pipeline:
#
# CLEAN TRAIN
#      │
#      ├── calculate reference correlations
#      └── calculate reference percentiles
#                  │
#                  ▼
#        ┌─────────────────────┐
#        │ Correlation anomaly │
#        │ injection           │
#        └─────────────────────┘
#             │      │      │
#             ▼      ▼      ▼
#           TRAIN   VAL     TEST
#
# Important:
# - Injection independently in TRAIN / VALIDATION / TEST
# - Physical / unscaled values
# - Reference statistics come ONLY from CLEAN TRAIN
# - No overlap with previous anomalies
# - used_rows_train / validation / test control overlap
# - No dependency on etiqueta_deteccion
# - No dependency on etiqueta_tipo_anomalia
# - Timestamp column: FECHA
# - Full traceability:
#       sensor_anomalia
#       anomalia_id
#       tipo_anomalia
#       fuente_anomalia
#       valor_original_anomalia
#       valor_inyectado_anomalia
# ============================================================


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# ============================================================
# 1. CONFIGURATION
# ============================================================

cfg_corr_dev = {
    "percentage_rows": 0.03,
    "filter_hour_start": 7,
    "filter_hour_end": 18,
}


# ============================================================
# PHYSICAL RELATIONSHIPS USED BY THE PROFESSOR
# ============================================================

PARES_DESVIACION_CORRELACION = [
    ("PRAD", "PRGINT"),   # exterior/interior solar radiation
    ("XTINV", "XHINV"),   # interior temperature/humidity
    ("PTEXT", "XTINV"),   # exterior/interior temperature
    ("PHEXT", "XHINV"),   # exterior/interior humidity
]


print("=" * 70)
print("PARES UTILIZADOS PARA DESVIACIÓN DE CORRELACIÓN")
print("=" * 70)

for sA, sB in PARES_DESVIACION_CORRELACION:
    print(f"  {sA:8s} <-> {sB}")


# ============================================================
# 2. REFERENCE CORRELATION MATRIX
#    CLEAN TRAIN ONLY
# ============================================================

print(
    "\nCalculando matriz de correlación "
    "sobre el TRAIN LIMPIO..."
)


# Include variables involved in the four pairs
# plus ventilation variables as contextual variables.

columnas_corr = list(
    dict.fromkeys(
        [
            c
            for par in PARES_DESVIACION_CORRELACION
            for c in par
        ]
        +
        [
            c
            for c in COLUMNAS_VENTILACION
            if c in df_train.columns
        ]
    )
)


# Keep only existing numeric columns

columnas_corr = [
    c
    for c in columnas_corr
    if (
        c in df_train.columns
        and pd.api.types.is_numeric_dtype(
            df_train[c]
        )
    )
]


if len(columnas_corr) < 2:

    raise ValueError(
        "No hay suficientes columnas numéricas "
        "para calcular la matriz de correlación."
    )


# IMPORTANT:
# Reference statistics come ONLY from CLEAN TRAIN.

correlation_matrix = (
    df_train[columnas_corr].corr()
)


print("\nMatriz de correlación — CLEAN TRAIN:")

print(
    correlation_matrix.round(3)
)


# ============================================================
# 3. VERIFY REQUIRED PAIR CORRELATIONS
# ============================================================

print(
    "\nCorrelaciones de referencia utilizadas:"
)

for sA, sB in PARES_DESVIACION_CORRELACION:

    if (
        sA not in correlation_matrix.index
        or
        sB not in correlation_matrix.columns
    ):

        raise ValueError(
            f"No existe correlación disponible "
            f"para {sA} <-> {sB}."
        )

    corr_value = correlation_matrix.loc[
        sA,
        sB
    ]

    print(
        f"  {sA:8s} <-> {sB:8s}: "
        f"{corr_value:.4f}"
    )


# ============================================================
# 4. OPTIONAL CORRELATION HEATMAP
# ============================================================

plt.figure(
    figsize=(12, 10)
)

sns.heatmap(
    correlation_matrix,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    linewidths=.5
)

plt.title(
    "Matriz de Correlación — Train limpio"
)

plt.tight_layout()

plt.show()


# ============================================================
# 5. REFERENCE PERCENTILES
#    CLEAN TRAIN ONLY
# ============================================================

columnas_para_percentiles = list(
    dict.fromkeys(
        [
            "PRAD",
            "PRGINT",
            "XTINV",
            "XHINV",
            "PTEXT",
            "PHEXT",
            "XCO2I",
            "UVENT_cen",
            "UVENT_lN",
        ]
    )
)


percentiles_calculados = {}


print(
    "\nCalculando percentiles "
    "sobre el TRAIN LIMPIO:"
)


for col in columnas_para_percentiles:

    if (
        col in df_train.columns
        and pd.api.types.is_numeric_dtype(
            df_train[col]
        )
    ):

        p25 = df_train[col].quantile(0.25)

        p50 = df_train[col].quantile(0.50)

        p75 = df_train[col].quantile(0.75)

        percentiles_calculados[col] = {
            "p25": p25,
            "p50": p50,
            "p75": p75,
        }

        print(
            f"  {col:10s}: "
            f"P25={p25:.3f}, "
            f"P50={p50:.3f}, "
            f"P75={p75:.3f}"
        )

    else:

        print(
            f"  Advertencia: {col} "
            f"no existe o no es numérica."
        )


# ============================================================
# 6. VALIDATE THAT ALL PAIR SENSORS HAVE PERCENTILES
# ============================================================

for sA, sB in PARES_DESVIACION_CORRELACION:

    if sA not in percentiles_calculados:

        raise ValueError(
            f"No hay percentiles de referencia "
            f"para {sA}."
        )

    if sB not in percentiles_calculados:

        raise ValueError(
            f"No hay percentiles de referencia "
            f"para {sB}."
        )


# ============================================================
# 7. INJECTION FUNCTION
# ============================================================

def inyectar_desviacion_correlacion(
    df,
    lista_pares,
    correlation_matrix,
    percentiles_ref,
    porcentaje_filas_afectadas,
    filtro_hora_inicio=7,
    filtro_hora_fin=18,
    random_state=42,
    used_rows=None,
    dataset_name="TRAIN",
):
    """
    Inject correlation-deviation anomalies independently
    into TRAIN / VALIDATION / TEST.

    IMPORTANT:
    - correlation_matrix MUST come from CLEAN TRAIN
    - percentiles_ref MUST come from CLEAN TRAIN
    - df is the current split after previous anomaly types
    - used_rows controls overlap inside this split

    For each anomaly:
        - select a physical pair (sA, sB)
        - keep sB unchanged
        - modify sA so it becomes inconsistent
          with the observed state of sB

    Returns:
        df_modificado
        used_rows
        df_eventos
    """

    # ========================================================
    # VALIDATION
    # ========================================================

    if not lista_pares:

        raise ValueError(
            "La lista de pares de correlación "
            "está vacía."
        )


    if not percentiles_ref:

        raise ValueError(
            "No se proporcionaron percentiles "
            "de referencia."
        )


    if correlation_matrix is None:

        raise ValueError(
            "No se proporcionó la matriz "
            "de correlación de referencia."
        )


    if "FECHA" not in df.columns:

        raise ValueError(
            "La columna 'FECHA' no existe "
            f"en {dataset_name}."
        )


    # ========================================================
    # RANDOM GENERATOR
    # ========================================================
    #
    # Dataset-specific seed keeps TRAIN / VAL / TEST
    # independent and reproducible.
    # ========================================================

    dataset_offsets = {
        "TRAIN": 0,
        "VALIDATION": 1000,
        "TEST": 2000,
    }

    seed = (
        int(random_state)
        +
        dataset_offsets.get(
            dataset_name,
            3000
        )
    )


    rng = np.random.default_rng(seed)


    # ========================================================
    # COPY DATA
    # ========================================================

    df_modificado = df.copy()


    # ========================================================
    # INITIALIZE / VALIDATE USED_ROWS
    # ========================================================

    if used_rows is None:

        used_rows = np.zeros(
            len(df_modificado),
            dtype=bool
        )

    else:

        used_rows = np.asarray(
            used_rows,
            dtype=bool
        ).copy()


        if len(used_rows) != len(
            df_modificado
        ):

            raise ValueError(
                f"used_rows no tiene la misma "
                f"longitud que {dataset_name}."
            )


    # ========================================================
    # ADD / VALIDATE TRACEABILITY COLUMNS
    # ========================================================

    if "sensor_anomalia" not in df_modificado.columns:

        df_modificado[
            "sensor_anomalia"
        ] = pd.NA


    if "anomalia_id" not in df_modificado.columns:

        df_modificado[
            "anomalia_id"
        ] = pd.NA


    if "tipo_anomalia" not in df_modificado.columns:

        df_modificado[
            "tipo_anomalia"
        ] = pd.NA


    if "fuente_anomalia" not in df_modificado.columns:

        df_modificado[
            "fuente_anomalia"
        ] = pd.NA


    if "valor_original_anomalia" not in df_modificado.columns:

        df_modificado[
            "valor_original_anomalia"
        ] = np.nan


    if "valor_inyectado_anomalia" not in df_modificado.columns:

        df_modificado[
            "valor_inyectado_anomalia"
        ] = np.nan


    # ========================================================
    # 6. TIME FILTER
    # ========================================================

    fechas = pd.to_datetime(
        df_modificado["FECHA"],
        errors="coerce"
    )


    horas = fechas.dt.hour


    if filtro_hora_inicio <= filtro_hora_fin:

        mask_hora = (
            (horas >= filtro_hora_inicio)
            &
            (horas <= filtro_hora_fin)
        )

    else:

        # Window crossing midnight

        mask_hora = (
            (horas >= filtro_hora_inicio)
            |
            (horas <= filtro_hora_fin)
        )


    # ========================================================
    # 7. AVAILABLE ROWS
    # ========================================================
    #
    # Previously injected rows are excluded.
    # `used_rows` is the only overlap-control mechanism.
    # ========================================================

    mask_available = ~used_rows


    mask_candidates = (
        mask_hora
        &
        mask_available
        &
        fechas.notna()
    )


    candidate_positions = np.flatnonzero(
        mask_candidates.to_numpy()
    )


    print(
        f"\n{dataset_name} — "
        "Candidatos disponibles para DesCor:",
        f"{len(candidate_positions):,}"
    )


    if len(candidate_positions) == 0:

        print(
            f"{dataset_name} — No hay filas "
            "candidatas disponibles."
        )

        return (
            df_modificado,
            used_rows,
            pd.DataFrame()
        )


    # ========================================================
    # 8. TARGET NUMBER OF ROWS
    # ========================================================

    objetivo_total = int(
        round(
            len(df_modificado)
            *
            porcentaje_filas_afectadas
        )
    )


    objetivo_total = min(
        objetivo_total,
        len(candidate_positions)
    )


    print(
        f"{dataset_name} — "
        f"Objetivo total DesCor: "
        f"{objetivo_total:,}"
    )


    # ========================================================
    # 9. RANDOMIZE CANDIDATES
    # ========================================================

    candidate_positions = rng.permutation(
        candidate_positions
    )


    # ========================================================
    # 10. EVENT RECORDS
    # ========================================================

    records = []


    pair_counts = {
        pair: 0
        for pair in lista_pares
    }


    rejected = 0


    # ========================================================
    # 11. HELPER — ONE INJECTION
    # ========================================================

    def intentar_inyectar(
        pos,
        sA,
        sB
    ):

        # ----------------------------------------------------
        # Validate reference information
        # ----------------------------------------------------

        if sA not in percentiles_ref:

            return False


        if sB not in percentiles_ref:

            return False


        if (
            sA not in correlation_matrix.index
            or
            sB not in correlation_matrix.columns
        ):

            return False


        # ----------------------------------------------------
        # Validate columns in current split
        # ----------------------------------------------------

        if (
            sA not in df_modificado.columns
            or
            sB not in df_modificado.columns
        ):

            return False


        # ----------------------------------------------------
        # Current values
        # ----------------------------------------------------

        vA = df_modificado.iloc[
            pos
        ][sA]

        vB = df_modificado.iloc[
            pos
        ][sB]


        # Cannot inject if either value is missing

        if (
            pd.isna(vA)
            or
            pd.isna(vB)
        ):

            return False


        # ----------------------------------------------------
        # Reference correlation
        # ----------------------------------------------------

        corr = correlation_matrix.loc[
            sA,
            sB
        ]


        if pd.isna(corr):

            return False


        # ----------------------------------------------------
        # Reference percentiles
        # ----------------------------------------------------

        pA = percentiles_ref[sA]

        pB = percentiles_ref[sB]


        nuevo_valor = None


        # ====================================================
        # POSITIVE CORRELATION
        # ====================================================

        if corr > 0.3:

            # B unusually HIGH
            # Normally A should also be high.
            # Make A unusually LOW.

            if vB > pB["p75"]:

                nuevo_valor = pA["p25"]


            # B unusually LOW
            # Normally A should also be low.
            # Make A unusually HIGH.

            elif vB < pB["p25"]:

                nuevo_valor = pA["p75"]


        # ====================================================
        # NEGATIVE CORRELATION
        # ====================================================

        elif corr < -0.3:

            # B unusually HIGH
            # Normally A should be LOW.
            # Make A HIGH.

            if vB > pB["p75"]:

                nuevo_valor = pA["p75"]


            # B unusually LOW
            # Normally A should be HIGH.
            # Make A LOW.

            elif vB < pB["p25"]:

                nuevo_valor = pA["p25"]


        # ====================================================
        # WEAK CORRELATION
        # ====================================================

        else:

            return False


        # ----------------------------------------------------
        # No valid anomaly generated
        # ----------------------------------------------------

        if nuevo_valor is None:

            return False


        # ----------------------------------------------------
        # Prevent unchanged value
        # ----------------------------------------------------

        if np.isclose(
            float(vA),
            float(nuevo_valor),
            equal_nan=False
        ):

            return False


        # ----------------------------------------------------
        # Save original value
        # ----------------------------------------------------

        valor_original = vA


        # ----------------------------------------------------
        # Modify sensor A
        # ----------------------------------------------------

        df_modificado.iat[
            pos,
            df_modificado.columns.get_loc(
                sA
            )
        ] = nuevo_valor


        # ----------------------------------------------------
        # Traceability
        # ----------------------------------------------------

        df_modificado.iat[
            pos,
            df_modificado.columns.get_loc(
                "sensor_anomalia"
            )
        ] = sA


        df_modificado.iat[
            pos,
            df_modificado.columns.get_loc(
                "anomalia_id"
            )
        ] = (
            f"DESCOR_{dataset_name}_"
            f"{sA}_{pos:06d}"
        )


        df_modificado.iat[
            pos,
            df_modificado.columns.get_loc(
                "tipo_anomalia"
            )
        ] = (
            "Desviacion de Correlacion"
        )


        df_modificado.iat[
            pos,
            df_modificado.columns.get_loc(
                "fuente_anomalia"
            )
        ] = (
            "professor_method_correlation"
        )


        df_modificado.iat[
            pos,
            df_modificado.columns.get_loc(
                "valor_original_anomalia"
            )
        ] = valor_original


        df_modificado.iat[
            pos,
            df_modificado.columns.get_loc(
                "valor_inyectado_anomalia"
            )
        ] = nuevo_valor


        return True


    # ========================================================
    # 12. INJECTION LOOP
    # ========================================================

    injected = 0


    for pos in candidate_positions:

        # Stop when target is reached

        if injected >= objetivo_total:

            break


        # ----------------------------------------------------
        # Select random physical pair
        # ----------------------------------------------------

        pair_idx = int(
            rng.integers(
                0,
                len(lista_pares)
            )
        )


        sA, sB = lista_pares[
            pair_idx
        ]


        # ----------------------------------------------------
        # Attempt injection
        # ----------------------------------------------------

        success = intentar_inyectar(
            pos,
            sA,
            sB
        )


        if success:

            # -----------------------------------------------
            # Mark row globally used
            # -----------------------------------------------

            used_rows[pos] = True


            injected += 1


            pair_counts[
                (sA, sB)
            ] += 1


            # -----------------------------------------------
            # Save event
            # -----------------------------------------------

            records.append({

                "dataset":
                    dataset_name,

                "row_position":
                    int(pos),

                "timestamp":
                    df_modificado.iloc[
                        pos
                    ]["FECHA"],

                "sensor_anomalia":
                    sA,

                "sensor_referencia":
                    sB,

                "anomalia_id":
                    df_modificado.iloc[
                        pos
                    ]["anomalia_id"],

                "tipo_anomalia":
                    df_modificado.iloc[
                        pos
                    ]["tipo_anomalia"],

                "fuente_anomalia":
                    df_modificado.iloc[
                        pos
                    ]["fuente_anomalia"],

                "valor_original":
                    df_modificado.iloc[
                        pos
                    ][
                        "valor_original_anomalia"
                    ],

                "valor_inyectado":
                    df_modificado.iloc[
                        pos
                    ][
                        "valor_inyectado_anomalia"
                    ],

                "correlacion_referencia":
                    correlation_matrix.loc[
                        sA,
                        sB
                    ],

            })


        else:

            rejected += 1


    # ========================================================
    # 13. EVENTS DATAFRAME
    # ========================================================

    df_eventos = pd.DataFrame(
        records
    )


    # ========================================================
    # 14. SUMMARY
    # ========================================================

    print(
        "\n" + "=" * 60
    )

    print(
        f"RESULTADO DESVIACIÓN DE CORRELACIÓN — "
        f"{dataset_name}"
    )

    print(
        "=" * 60
    )


    print(
        f"Objetivo:      {objetivo_total:,}"
    )


    print(
        f"Inyectadas:    {injected:,}"
    )


    print(
        f"Rechazadas:    {rejected:,}"
    )


    print(
        f"Filas usadas:  {used_rows.sum():,}"
    )


    print(
        "\nInyecciones por par:"
    )


    for pair, count in pair_counts.items():

        print(
            f"  {pair[0]:8s} <-> "
            f"{pair[1]:8s}: "
            f"{count:,}"
        )


    # Warn if target was not reached

    if injected < objetivo_total:

        print(
            "\nWARNING:"
        )

        print(
            f"Only {injected:,} of "
            f"{objetivo_total:,} requested "
            "correlation anomalies were generated."
        )


    return (
        df_modificado,
        used_rows,
        df_eventos
    )


# ============================================================
# 8. RUN — TRAIN
# ============================================================

print(
    "\n" + "=" * 70
)

print(
    "INICIANDO DESVIACIÓN DE CORRELACIÓN — TRAIN"
)

print(
    "=" * 70
)


(
    df_train_after_corr,
    used_rows_train,
    df_eventos_corr_train
) = inyectar_desviacion_correlacion(

    df=df_train_after_oor,

    lista_pares=
        PARES_DESVIACION_CORRELACION,

    correlation_matrix=
        correlation_matrix,

    percentiles_ref=
        percentiles_calculados,

    porcentaje_filas_afectadas=
        cfg_corr_dev[
            "percentage_rows"
        ],

    filtro_hora_inicio=
        cfg_corr_dev[
            "filter_hour_start"
        ],

    filtro_hora_fin=
        cfg_corr_dev[
            "filter_hour_end"
        ],

    random_state=
        RANDOM_STATE,

    used_rows=
        used_rows_train,

    dataset_name="TRAIN",
)


# ============================================================
# 9. RUN — VALIDATION
# ============================================================

print(
    "\n" + "=" * 70
)

print(
    "INICIANDO DESVIACIÓN DE CORRELACIÓN — VALIDATION"
)

print(
    "=" * 70
)


(
    df_validation_after_corr,
    used_rows_validation,
    df_eventos_corr_validation
) = inyectar_desviacion_correlacion(

    df=df_validation_after_oor,

    lista_pares=
        PARES_DESVIACION_CORRELACION,

    # IMPORTANT:
    # Reference statistics still come from CLEAN TRAIN.

    correlation_matrix=
        correlation_matrix,

    percentiles_ref=
        percentiles_calculados,

    porcentaje_filas_afectadas=
        cfg_corr_dev[
            "percentage_rows"
        ],

    filtro_hora_inicio=
        cfg_corr_dev[
            "filter_hour_start"
        ],

    filtro_hora_fin=
        cfg_corr_dev[
            "filter_hour_end"
        ],

    random_state=
        RANDOM_STATE,

    used_rows=
        used_rows_validation,

    dataset_name="VALIDATION",
)


# ============================================================
# 10. RUN — TEST
# ============================================================

print(
    "\n" + "=" * 70
)

print(
    "INICIANDO DESVIACIÓN DE CORRELACIÓN — TEST"
)

print(
    "=" * 70
)


(
    df_test_after_corr,
    used_rows_test,
    df_eventos_corr_test
) = inyectar_desviacion_correlacion(

    df=df_test_after_oor,

    lista_pares=
        PARES_DESVIACION_CORRELACION,

    # IMPORTANT:
    # Reference statistics still come from CLEAN TRAIN.

    correlation_matrix=
        correlation_matrix,

    percentiles_ref=
        percentiles_calculados,

    porcentaje_filas_afectadas=
        cfg_corr_dev[
            "percentage_rows"
        ],

    filtro_hora_inicio=
        cfg_corr_dev[
            "filter_hour_start"
        ],

    filtro_hora_fin=
        cfg_corr_dev[
            "filter_hour_end"
        ],

    random_state=
        RANDOM_STATE,

    used_rows=
        used_rows_test,

    dataset_name="TEST",
)


# ============================================================
# 11. GENERAL AUDIT FUNCTION
# ============================================================

def audit_desviacion_correlacion(
    df,
    df_eventos,
    used_rows,
    dataset_name
):

    print(
        "\n" + "=" * 70
    )

    print(
        f"{dataset_name} — "
        "DESVIACIÓN DE CORRELACIÓN AUDIT"
    )

    print(
        "=" * 70
    )


    # --------------------------------------------------------
    # Rows with this anomaly
    # --------------------------------------------------------

    mask_corr = (
        df[
            "tipo_anomalia"
        ]
        .eq(
            "Desviacion de Correlacion"
        )
    )


    df_check = df.loc[
        mask_corr
    ]


    print(
        f"\nRows DesCor: "
        f"{len(df_check):,}"
    )


    print(
        f"Events registered: "
        f"{len(df_eventos):,}"
    )


    # --------------------------------------------------------
    # Traceability
    # --------------------------------------------------------

    metadata_columns = [
        "sensor_anomalia",
        "anomalia_id",
        "tipo_anomalia",
        "fuente_anomalia",
        "valor_original_anomalia",
        "valor_inyectado_anomalia",
    ]


    print(
        "\nTraceability:"
    )


    for col in metadata_columns:

        missing = (
            df_check[col]
            .isna()
            .sum()
        )

        print(
            f"{col:32s}: "
            f"{missing} missing"
        )

        assert missing == 0


    # --------------------------------------------------------
    # Correct anomaly type
    # --------------------------------------------------------

    wrong_type = (
        df_check[
            "tipo_anomalia"
        ]
        .ne(
            "Desviacion de Correlacion"
        )
        .sum()
    )


    print(
        "\nWrong anomaly type:",
        wrong_type
    )


    assert wrong_type == 0


    # --------------------------------------------------------
    # Correct source
    # --------------------------------------------------------

    expected_source = (
        "professor_method_correlation"
    )


    wrong_source = (
        df_check[
            "fuente_anomalia"
        ]
        .ne(
            expected_source
        )
        .sum()
    )


    print(
        "Wrong anomaly source:",
        wrong_source
    )


    assert wrong_source == 0


    # --------------------------------------------------------
    # Sensor distribution
    # --------------------------------------------------------

    print(
        "\nAffected sensors:"
    )


    if len(df_check) > 0:

        print(
            df_check[
                "sensor_anomalia"
            ].value_counts()
        )


    # --------------------------------------------------------
    # Unique IDs
    # --------------------------------------------------------

    unique_ids = (
        df_check[
            "anomalia_id"
        ]
        .nunique()
    )


    duplicate_ids = (
        len(df_check)
        -
        unique_ids
    )


    print(
        "\nUnique anomaly IDs:",
        unique_ids
    )


    print(
        "Duplicate IDs:",
        duplicate_ids
    )


    assert duplicate_ids == 0


    # --------------------------------------------------------
    # Original != injected
    # --------------------------------------------------------

    if len(df_check) > 0:

        same_values = np.isclose(
            df_check[
                "valor_original_anomalia"
            ].astype(float),

            df_check[
                "valor_inyectado_anomalia"
            ].astype(float),

            equal_nan=False
        )


        same_count = int(
            same_values.sum()
        )

    else:

        same_count = 0


    print(
        "\nOriginal == injected:",
        same_count
    )


    assert same_count == 0


    # --------------------------------------------------------
    # Verify events are marked in used_rows
    # --------------------------------------------------------

    if len(df_eventos) > 0:

        posiciones = (
            df_eventos[
                "row_position"
            ]
            .astype(int)
            .to_numpy()
        )


        no_marcadas = int(
            np.sum(
                ~used_rows[
                    posiciones
                ]
            )
        )

    else:

        no_marcadas = 0


    print(
        "Events not marked in used_rows:",
        no_marcadas
    )


    assert no_marcadas == 0


    # --------------------------------------------------------
    # Verify no duplicate positions
    # --------------------------------------------------------

    if len(df_eventos) > 0:

        posiciones = (
            df_eventos[
                "row_position"
            ]
            .astype(int)
            .to_numpy()
        )


        duplicate_positions = (
            len(posiciones)
            -
            len(np.unique(posiciones))
        )

    else:

        duplicate_positions = 0


    print(
        "Duplicate positions:",
        duplicate_positions
    )


    assert duplicate_positions == 0


    # --------------------------------------------------------
    # Verify event count matches dataframe count
    # --------------------------------------------------------

    event_count = len(
        df_eventos
    )

    row_count = len(
        df_check
    )


    print(
        "\nRows vs events:",
        row_count,
        "vs",
        event_count
    )


    assert row_count == event_count


    # --------------------------------------------------------
    # Pair distribution
    # --------------------------------------------------------

    print(
        "\nInjections by pair:"
    )


    if len(df_eventos) > 0:

        print(
            df_eventos[
                [
                    "sensor_anomalia",
                    "sensor_referencia"
                ]
            ]
            .value_counts()
        )


    # --------------------------------------------------------
    # Final result
    # --------------------------------------------------------

    print(
        "\n" + "-" * 70
    )

    print(
        f"✓ {dataset_name} "
        "DesCor audit passed."
    )

    print(
        "-" * 70
    )


# ============================================================
# 12. RUN AUDITS
# ============================================================

audit_desviacion_correlacion(
    df=df_train_after_corr,
    df_eventos=df_eventos_corr_train,
    used_rows=used_rows_train,
    dataset_name="TRAIN",
)


audit_desviacion_correlacion(
    df=df_validation_after_corr,
    df_eventos=df_eventos_corr_validation,
    used_rows=used_rows_validation,
    dataset_name="VALIDATION",
)


audit_desviacion_correlacion(
    df=df_test_after_corr,
    df_eventos=df_eventos_corr_test,
    used_rows=used_rows_test,
    dataset_name="TEST",
)


# ============================================================
# 13. FINAL SUMMARY
# ============================================================

print(
    "\n" + "=" * 70
)

print(
    "DESVIACIÓN DE CORRELACIÓN — FINAL SUMMARY"
)

print(
    "=" * 70
)


print(
    f"TRAIN      : "
    f"{len(df_eventos_corr_train):,} anomalies"
)


print(
    f"VALIDATION : "
    f"{len(df_eventos_corr_validation):,} anomalies"
)


print(
    f"TEST       : "
    f"{len(df_eventos_corr_test):,} anomalies"
)


print(
    "\nReference statistics:"
)

print(
    "  ✓ Correlation matrix: CLEAN TRAIN"
)

print(
    "  ✓ Percentiles: CLEAN TRAIN"
)

print(
    "\nOverlap control:"
)

print(
    "  ✓ used_rows_train"
)

print(
    "  ✓ used_rows_validation"
)

print(
    "  ✓ used_rows_test"
)

print(
    "\n✓ Desviación de Correlación completed."
)

print(
    "=" * 70
)

In [ ]:
# ============================================================
# CONTEXTUAL ANOMALIES
# TRAIN + VALIDATION + TEST
# ============================================================
#
# Pipeline:
#
# TRAIN:
#   df_train_after_corr
#       ↓
#   Contextual 1
#       ↓
#   Contextual 2
#       ↓
#   Contextual 3
#
# VALIDATION:
#   df_validation_after_corr
#       ↓
#   Contextual 1
#       ↓
#   Contextual 2
#       ↓
#   Contextual 3
#
# TEST:
#   df_test_after_corr
#       ↓
#   Contextual 1
#       ↓
#   Contextual 2
#       ↓
#   Contextual 3
#
# Important:
# - Physical / unscaled values
# - Same professor methodology
# - Applied independently to TRAIN / VALIDATION / TEST
# - No overlap with previous anomalies
# - used_rows is cumulative within each split
# - No dependency on etiqueta_deteccion
# - No dependency on etiqueta_tipo_anomalia
# - Timestamp column: FECHA
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 0. CONFIGURATION
# ============================================================

cfg_contextual_prad = {
    "percentage_rows": 0.02,
    "prad_min": 200.0,
    "ratio_normal_min": 0.40,
    "ratio_anomalo_max": 0.08,
}


cfg_contextual_co2 = {
    "num_sequences": 11,
    "umbral_vent_abierta": 40.0,
    "prad_min_diurno": 50.0,
    "co2_anomalo_min": 700.0,
    "co2_anomalo_max": 1200.0,
    "duracion_min": 3,
    "duracion_max": 8,
}


cfg_contextual_temp = {
    "percentage_rows": 0.016,
    "prad_min": 400.0,
    "ptext_min": 20.0,
    "delta_min": 3.0,
    "delta_max": 10.0,
}


COLUMNAS_VENTILACION = [
    "UVENT_cen",
    "UVENT_lN",
]


# ============================================================
# TRACEABILITY COLUMNS
# ============================================================

def _ensure_contextual_traceability_columns(df):

    df = df.copy()

    if "sensor_anomalia" not in df.columns:
        df["sensor_anomalia"] = pd.NA

    if "anomalia_id" not in df.columns:
        df["anomalia_id"] = pd.NA

    if "tipo_anomalia" not in df.columns:
        df["tipo_anomalia"] = pd.NA

    if "fuente_anomalia" not in df.columns:
        df["fuente_anomalia"] = pd.NA

    if "valor_original_anomalia" not in df.columns:
        df["valor_original_anomalia"] = np.nan

    if "valor_inyectado_anomalia" not in df.columns:
        df["valor_inyectado_anomalia"] = np.nan

    return df


# ============================================================
# 1. CONTEXTUAL TYPE 1
# PRAD HIGH / PRGINT ANOMALOUSLY LOW
# ============================================================

def inyectar_prad_prgint_contextual(
    df,
    porcentaje_filas_afectadas,
    prad_min=200.0,
    ratio_normal_min=0.40,
    ratio_anomalo_max=0.08,
    random_state=42,
    used_rows=None,
    dataset_name="TRAIN",
):
    """
    Contextual anomaly:

        PRAD high
        +
        PRGINT normally high relative to PRAD
        ->
        force PRGINT to an anomalously low fraction of PRAD.

    Example:
        PRAD = 600 W/m2
        normal PRGINT/PRAD > 0.40
        anomaly PRGINT/PRAD <= 0.08
    """

    df_modificado = _ensure_contextual_traceability_columns(df)

    rng = np.random.default_rng(
        random_state
    )

    n_rows = len(df_modificado)

    # --------------------------------------------------------
    # USED ROWS
    # --------------------------------------------------------

    if used_rows is None:

        used_rows = np.zeros(
            n_rows,
            dtype=bool
        )

    else:

        used_rows = np.asarray(
            used_rows,
            dtype=bool
        ).copy()

        if len(used_rows) != n_rows:

            raise ValueError(
                f"{dataset_name}: used_rows no tiene "
                "la misma longitud que el DataFrame."
            )

    # --------------------------------------------------------
    # VALUES
    # --------------------------------------------------------

    prad_vals = pd.to_numeric(
        df_modificado["PRAD"],
        errors="coerce"
    ).to_numpy(dtype=float)

    prgint_vals = pd.to_numeric(
        df_modificado["PRGINT"],
        errors="coerce"
    ).to_numpy(dtype=float)

    ratio_actual = np.where(
        prad_vals > 1.0,
        prgint_vals / (prad_vals + 1e-3),
        np.nan,
    )

    # --------------------------------------------------------
    # CANDIDATES
    # --------------------------------------------------------

    mask_candidates = (
        ~used_rows
        & np.isfinite(prad_vals)
        & np.isfinite(prgint_vals)
        & (prad_vals > prad_min)
        & (ratio_actual > ratio_normal_min)
    )

    idx_candidates = np.flatnonzero(
        mask_candidates
    )

    print("\n" + "=" * 60)
    print(
        f"{dataset_name} — CONTEXTUAL 1: "
        "PRAD / PRGINT"
    )
    print("=" * 60)

    print(
        f"Candidatos disponibles: "
        f"{len(idx_candidates):,}"
    )

    n_target = int(
        round(
            n_rows *
            porcentaje_filas_afectadas
        )
    )

    print(
        f"Objetivo: {n_target:,} filas"
    )

    if len(idx_candidates) == 0:

        print("Sin candidatos.")

        return (
            df_modificado,
            used_rows,
            pd.DataFrame()
        )

    n = min(
        n_target,
        len(idx_candidates)
    )

    selected = rng.choice(
        idx_candidates,
        size=n,
        replace=False,
    )

    prgint_col = (
        df_modificado.columns.get_loc(
            "PRGINT"
        )
    )

    events = []

    for event_number, pos in enumerate(
        selected,
        start=1
    ):

        original_value = float(
            df_modificado.iat[
                pos,
                prgint_col
            ]
        )

        prad_value = float(
            prad_vals[pos]
        )

        ratio = float(
            rng.uniform(
                0.02,
                ratio_anomalo_max,
            )
        )

        injected_value = (
            prad_value * ratio
        )

        # Safety: must actually change value
        if np.isclose(
            original_value,
            injected_value
        ):
            continue

        df_modificado.iat[
            pos,
            prgint_col
        ] = injected_value

        used_rows[pos] = True

        anomaly_id = (
            f"CONTEXTUAL_{dataset_name}_"
            f"PRAD_PRGINT_{event_number:06d}"
        )

        # Traceability
        df_modificado.iat[
            pos,
            df_modificado.columns.get_loc(
                "sensor_anomalia"
            )
        ] = "PRGINT"

        df_modificado.iat[
            pos,
            df_modificado.columns.get_loc(
                "anomalia_id"
            )
        ] = anomaly_id

        df_modificado.iat[
            pos,
            df_modificado.columns.get_loc(
                "tipo_anomalia"
            )
        ] = "Contextual"

        df_modificado.iat[
            pos,
            df_modificado.columns.get_loc(
                "fuente_anomalia"
            )
        ] = "professor_method_contextual"

        df_modificado.iat[
            pos,
            df_modificado.columns.get_loc(
                "valor_original_anomalia"
            )
        ] = original_value

        df_modificado.iat[
            pos,
            df_modificado.columns.get_loc(
                "valor_inyectado_anomalia"
            )
        ] = injected_value

        events.append({

            "dataset":
                dataset_name,

            "row_position":
                int(pos),

            "timestamp":
                df_modificado.iloc[pos]["FECHA"]
                if "FECHA" in df_modificado.columns
                else pd.NaT,

            "sensor_anomalia":
                "PRGINT",

            "tipo_anomalia":
                "Contextual",

            "fuente_anomalia":
                "professor_method_contextual",

            "contexto":
                "PRAD_PRGINT",

            "anomalia_id":
                anomaly_id,

            "valor_original":
                original_value,

            "valor_inyectado":
                injected_value,

            "PRAD":
                prad_value,

            "ratio_PRGINT_PRAD":
                ratio,
        })

    df_eventos = pd.DataFrame(events)

    print(
        f"Inyectadas: {len(df_eventos):,}"
    )

    return (
        df_modificado,
        used_rows,
        df_eventos
    )


# ============================================================
# 2. CONTEXTUAL TYPE 2
# HIGH CO2 WITH VENTILATION OPEN
# ============================================================

def inyectar_co2_ventilacion_contextual(
    df,
    num_secuencias_afectadas,
    umbral_vent_abierta=40.0,
    prad_min_diurno=50.0,
    co2_anomalo_min=700.0,
    co2_anomalo_max=1200.0,
    duracion_anomalia_min=3,
    duracion_anomalia_max=8,
    random_state=42,
    used_rows=None,
    dataset_name="TRAIN",
):
    """
    Contextual anomaly:

        ventilation open
        +
        daytime radiation
        +
        high interior CO2

    Creates contiguous anomaly sequences of 3-8 samples.
    """

    df_modificado = _ensure_contextual_traceability_columns(df)

    rng = np.random.default_rng(
        random_state
    )

    n = len(df_modificado)

    if used_rows is None:

        used_rows = np.zeros(
            n,
            dtype=bool
        )

    else:

        used_rows = np.asarray(
            used_rows,
            dtype=bool
        ).copy()

        if len(used_rows) != n:

            raise ValueError(
                f"{dataset_name}: used_rows no tiene "
                "la misma longitud que el DataFrame."
            )

    print("\n" + "=" * 60)
    print(
        f"{dataset_name} — CONTEXTUAL 2: "
        "CO2 + VENTILATION"
    )
    print("=" * 60)

    # --------------------------------------------------------
    # VENTILATION
    # --------------------------------------------------------

    ventilation_columns = [
        c
        for c in COLUMNAS_VENTILACION
        if c in df_modificado.columns
    ]

    if not ventilation_columns:

        print(
            "No ventilation columns available."
        )

        return (
            df_modificado,
            used_rows,
            pd.DataFrame()
        )

    # --------------------------------------------------------
    # VALUES
    # --------------------------------------------------------

    vent_media = (
        df_modificado[
            ventilation_columns
        ]
        .mean(
            axis=1,
            skipna=True
        )
        .to_numpy(dtype=float)
    )

    prad_vals = pd.to_numeric(
        df_modificado["PRAD"],
        errors="coerce"
    ).to_numpy(dtype=float)

    xco2i_vals = pd.to_numeric(
        df_modificado["XCO2I"],
        errors="coerce"
    ).to_numpy(dtype=float)

    xco2i_col = (
        df_modificado.columns.get_loc(
            "XCO2I"
        )
    )

    # --------------------------------------------------------
    # CANDIDATE STARTS
    # --------------------------------------------------------

    candidate_starts = []

    max_start = (
        n -
        duracion_anomalia_max
    )

    for start in range(
        max_start
    ):

        end = (
            start +
            duracion_anomalia_max
        )

        if np.any(
            used_rows[start:end]
        ):
            continue

        if not np.all(
            np.isfinite(
                vent_media[start:end]
            )
            &
            np.isfinite(
                prad_vals[start:end]
            )
            &
            np.isfinite(
                xco2i_vals[start:end]
            )
        ):
            continue

        if not np.all(
            vent_media[start:end]
            >=
            umbral_vent_abierta
        ):
            continue

        if not np.all(
            prad_vals[start:end]
            >=
            prad_min_diurno
        ):
            continue

        candidate_starts.append(
            start
        )

    candidate_starts = np.asarray(
        candidate_starts,
        dtype=int
    )

    print(
        f"Candidatos de secuencia: "
        f"{len(candidate_starts):,}"
    )

    if len(candidate_starts) == 0:

        print("Sin candidatos.")

        return (
            df_modificado,
            used_rows,
            pd.DataFrame()
        )

    # --------------------------------------------------------
    # CREATE SEQUENCES
    # --------------------------------------------------------

    events = []

    available_starts = (
        candidate_starts.copy()
    )

    sequences_created = 0

    max_attempts = max(
        num_secuencias_afectadas * 50,
        100,
    )

    attempts = 0

    while (
        sequences_created
        <
        num_secuencias_afectadas
        and
        len(available_starts)
        >
        0
        and
        attempts
        <
        max_attempts
    ):

        attempts += 1

        duration = int(
            rng.integers(
                duracion_anomalia_min,
                duracion_anomalia_max + 1,
            )
        )

        valid_starts = []

        for start in available_starts:

            end = (
                start +
                duration
            )

            if end > n:
                continue

            if np.any(
                used_rows[start:end]
            ):
                continue

            if not np.all(
                np.isfinite(
                    vent_media[start:end]
                )
                &
                np.isfinite(
                    prad_vals[start:end]
                )
                &
                np.isfinite(
                    xco2i_vals[start:end]
                )
            ):
                continue

            if not np.all(
                vent_media[start:end]
                >=
                umbral_vent_abierta
            ):
                continue

            if not np.all(
                prad_vals[start:end]
                >=
                prad_min_diurno
            ):
                continue

            valid_starts.append(
                start
            )

        if not valid_starts:
            break

        start = int(
            rng.choice(
                valid_starts
            )
        )

        end = (
            start +
            duration
        )

        co2_value = float(
            rng.uniform(
                co2_anomalo_min,
                co2_anomalo_max,
            )
        )

        sequence_number = (
            sequences_created + 1
        )

        sequence_id = (
            f"CONTEXTUAL_{dataset_name}_"
            f"CO2_VENT_{sequence_number:04d}"
        )

        for pos in range(
            start,
            end
        ):

            original_value = float(
                df_modificado.iat[
                    pos,
                    xco2i_col
                ]
            )

            injected_value = (
                co2_value
                +
                float(
                    rng.uniform(
                        -10.0,
                        10.0
                    )
                )
            )

            df_modificado.iat[
                pos,
                xco2i_col
            ] = injected_value

            used_rows[pos] = True

            # Traceability
            df_modificado.iat[
                pos,
                df_modificado.columns.get_loc(
                    "sensor_anomalia"
                )
            ] = "XCO2I"

            df_modificado.iat[
                pos,
                df_modificado.columns.get_loc(
                    "anomalia_id"
                )
            ] = (
                f"{sequence_id}_{pos:06d}"
            )

            df_modificado.iat[
                pos,
                df_modificado.columns.get_loc(
                    "tipo_anomalia"
                )
            ] = "Contextual"

            df_modificado.iat[
                pos,
                df_modificado.columns.get_loc(
                    "fuente_anomalia"
                )
            ] = (
                "professor_method_contextual"
            )

            df_modificado.iat[
                pos,
                df_modificado.columns.get_loc(
                    "valor_original_anomalia"
                )
            ] = original_value

            df_modificado.iat[
                pos,
                df_modificado.columns.get_loc(
                    "valor_inyectado_anomalia"
                )
            ] = injected_value

            events.append({

                "dataset":
                    dataset_name,

                "row_position":
                    int(pos),

                "timestamp":
                    df_modificado.iloc[pos]["FECHA"]
                    if "FECHA"
                    in df_modificado.columns
                    else pd.NaT,

                "sensor_anomalia":
                    "XCO2I",

                "tipo_anomalia":
                    "Contextual",

                "fuente_anomalia":
                    "professor_method_contextual",

                "contexto":
                    "CO2_VENTILACION",

                "secuencia":
                    sequence_number,

                "sequence_id":
                    sequence_id,

                "anomalia_id":
                    f"{sequence_id}_{pos:06d}",

                "valor_original":
                    original_value,

                "valor_inyectado":
                    injected_value,

                "vent_media":
                    float(
                        vent_media[pos]
                    ),

                "PRAD":
                    float(
                        prad_vals[pos]
                    ),
            })

        sequences_created += 1

        # Remove starts that overlap
        available_starts = (
            available_starts[
                (
                    available_starts
                    >=
                    end
                )
                |
                (
                    available_starts
                    +
                    duracion_anomalia_max
                    <=
                    start
                )
            ]
        )

    print(
        f"Secuencias creadas: "
        f"{sequences_created}/"
        f"{num_secuencias_afectadas}"
    )

    if (
        sequences_created
        <
        num_secuencias_afectadas
    ):

        print(
            "Aviso: no se pudieron crear "
            "todas las secuencias solicitadas."
        )

    df_eventos = pd.DataFrame(
        events
    )

    return (
        df_modificado,
        used_rows,
        df_eventos
    )


# ============================================================
# 3. CONTEXTUAL TYPE 3
# TEMPERATURE INVERSION
# ============================================================

def inyectar_temp_inversion_contextual(
    df,
    porcentaje_filas_afectadas,
    prad_min=400.0,
    ptext_min=20.0,
    delta_min=3.0,
    delta_max=10.0,
    random_state=42,
    used_rows=None,
    dataset_name="TRAIN",
):
    """
    Contextual anomaly:

        high radiation
        +
        high exterior temperature
        +
        XTINV >= PTEXT

    Then force:

        XTINV = PTEXT - random(3, 10)
    """

    df_modificado = _ensure_contextual_traceability_columns(df)

    rng = np.random.default_rng(
        random_state
    )

    n_rows = len(df_modificado)

    if used_rows is None:

        used_rows = np.zeros(
            n_rows,
            dtype=bool
        )

    else:

        used_rows = np.asarray(
            used_rows,
            dtype=bool
        ).copy()

        if len(used_rows) != n_rows:

            raise ValueError(
                f"{dataset_name}: used_rows no tiene "
                "la misma longitud que el DataFrame."
            )

    prad_vals = pd.to_numeric(
        df_modificado["PRAD"],
        errors="coerce"
    ).to_numpy(dtype=float)

    ptext_vals = pd.to_numeric(
        df_modificado["PTEXT"],
        errors="coerce"
    ).to_numpy(dtype=float)

    xtinv_vals = pd.to_numeric(
        df_modificado["XTINV"],
        errors="coerce"
    ).to_numpy(dtype=float)

    mask_candidates = (
        ~used_rows
        & np.isfinite(prad_vals)
        & np.isfinite(ptext_vals)
        & np.isfinite(xtinv_vals)
        & (prad_vals > prad_min)
        & (ptext_vals > ptext_min)
        & (xtinv_vals >= ptext_vals)
    )

    idx_candidates = np.flatnonzero(
        mask_candidates
    )

    print("\n" + "=" * 60)
    print(
        f"{dataset_name} — CONTEXTUAL 3: "
        "TEMPERATURE INVERSION"
    )
    print("=" * 60)

    print(
        f"Candidatos disponibles: "
        f"{len(idx_candidates):,}"
    )

    n_target = int(
        round(
            n_rows *
            porcentaje_filas_afectadas
        )
    )

    print(
        f"Objetivo: {n_target:,} filas"
    )

    if len(idx_candidates) == 0:

        print("Sin candidatos.")

        return (
            df_modificado,
            used_rows,
            pd.DataFrame()
        )

    n = min(
        n_target,
        len(idx_candidates)
    )

    selected = rng.choice(
        idx_candidates,
        size=n,
        replace=False,
    )

    xtinv_col = (
        df_modificado.columns.get_loc(
            "XTINV"
        )
    )

    events = []

    deltas = rng.uniform(
        delta_min,
        delta_max,
        size=n,
    )

    event_counter = 0

    for i, pos in enumerate(
        selected
    ):

        ptext_value = float(
            df_modificado.iat[
                pos,
                df_modificado.columns.get_loc(
                    "PTEXT"
                )
            ]
        )

        original_value = float(
            df_modificado.iat[
                pos,
                xtinv_col
            ]
        )

        injected_value = (
            ptext_value -
            deltas[i]
        )

        if np.isclose(
            original_value,
            injected_value
        ):
            continue

        df_modificado.iat[
            pos,
            xtinv_col
        ] = injected_value

        used_rows[pos] = True

        event_counter += 1

        anomaly_id = (
            f"CONTEXTUAL_{dataset_name}_"
            f"TEMP_INV_{event_counter:06d}"
        )

        # Traceability
        df_modificado.iat[
            pos,
            df_modificado.columns.get_loc(
                "sensor_anomalia"
            )
        ] = "XTINV"

        df_modificado.iat[
            pos,
            df_modificado.columns.get_loc(
                "anomalia_id"
            )
        ] = anomaly_id

        df_modificado.iat[
            pos,
            df_modificado.columns.get_loc(
                "tipo_anomalia"
            )
        ] = "Contextual"

        df_modificado.iat[
            pos,
            df_modificado.columns.get_loc(
                "fuente_anomalia"
            )
        ] = (
            "professor_method_contextual"
        )

        df_modificado.iat[
            pos,
            df_modificado.columns.get_loc(
                "valor_original_anomalia"
            )
        ] = original_value

        df_modificado.iat[
            pos,
            df_modificado.columns.get_loc(
                "valor_inyectado_anomalia"
            )
        ] = injected_value

        events.append({

            "dataset":
                dataset_name,

            "row_position":
                int(pos),

            "timestamp":
                df_modificado.iloc[pos]["FECHA"]
                if "FECHA"
                in df_modificado.columns
                else pd.NaT,

            "sensor_anomalia":
                "XTINV",

            "tipo_anomalia":
                "Contextual",

            "fuente_anomalia":
                "professor_method_contextual",

            "contexto":
                "TEMP_INVERSION",

            "anomalia_id":
                anomaly_id,

            "valor_original":
                original_value,

            "valor_inyectado":
                injected_value,

            "PTEXT":
                ptext_value,

            "delta":
                float(deltas[i]),

            "PRAD":
                float(prad_vals[pos]),
        })

    df_eventos = pd.DataFrame(
        events
    )

    print(
        f"Inyectadas: "
        f"{len(df_eventos):,}"
    )

    return (
        df_modificado,
        used_rows,
        df_eventos
    )


# ============================================================
# 4. RUN ONE COMPLETE DATASET
# ============================================================

def aplicar_anomalias_contextuales(
    df,
    used_rows,
    dataset_name,
    random_state=42,
):

    print("\n")
    print("#" * 70)
    print(
        f"INICIANDO ANOMALÍAS CONTEXTUALES — "
        f"{dataset_name}"
    )
    print("#" * 70)

    # --------------------------------------------------------
    # CONTEXTUAL 1
    # --------------------------------------------------------

    (
        df_after_1,
        used_rows,
        events_1,
    ) = inyectar_prad_prgint_contextual(

        df=df,

        porcentaje_filas_afectadas=
            cfg_contextual_prad[
                "percentage_rows"
            ],

        prad_min=
            cfg_contextual_prad[
                "prad_min"
            ],

        ratio_normal_min=
            cfg_contextual_prad[
                "ratio_normal_min"
            ],

        ratio_anomalo_max=
            cfg_contextual_prad[
                "ratio_anomalo_max"
            ],

        random_state=
            random_state,

        used_rows=
            used_rows,

        dataset_name=
            dataset_name,
    )

    # --------------------------------------------------------
    # CONTEXTUAL 2
    # --------------------------------------------------------

    (
        df_after_2,
        used_rows,
        events_2,
    ) = inyectar_co2_ventilacion_contextual(

        df=df_after_1,

        num_secuencias_afectadas=
            cfg_contextual_co2[
                "num_sequences"
            ],

        umbral_vent_abierta=
            cfg_contextual_co2[
                "umbral_vent_abierta"
            ],

        prad_min_diurno=
            cfg_contextual_co2[
                "prad_min_diurno"
            ],

        co2_anomalo_min=
            cfg_contextual_co2[
                "co2_anomalo_min"
            ],

        co2_anomalo_max=
            cfg_contextual_co2[
                "co2_anomalo_max"
            ],

        duracion_anomalia_min=
            cfg_contextual_co2[
                "duracion_min"
            ],

        duracion_anomalia_max=
            cfg_contextual_co2[
                "duracion_max"
            ],

        random_state=
            random_state,

        used_rows=
            used_rows,

        dataset_name=
            dataset_name,
    )

    # --------------------------------------------------------
    # CONTEXTUAL 3
    # --------------------------------------------------------

    (
        df_after_3,
        used_rows,
        events_3,
    ) = inyectar_temp_inversion_contextual(

        df=df_after_2,

        porcentaje_filas_afectadas=
            cfg_contextual_temp[
                "percentage_rows"
            ],

        prad_min=
            cfg_contextual_temp[
                "prad_min"
            ],

        ptext_min=
            cfg_contextual_temp[
                "ptext_min"
            ],

        delta_min=
            cfg_contextual_temp[
                "delta_min"
            ],

        delta_max=
            cfg_contextual_temp[
                "delta_max"
            ],

        random_state=
            random_state,

        used_rows=
            used_rows,

        dataset_name=
            dataset_name,
    )

    # --------------------------------------------------------
    # COMBINE EVENTS
    # --------------------------------------------------------

    event_tables = [
        events_1,
        events_2,
        events_3,
    ]

    event_tables = [
        x
        for x in event_tables
        if x is not None
        and not x.empty
    ]

    if event_tables:

        df_eventos = pd.concat(
            event_tables,
            ignore_index=True,
        )

    else:

        df_eventos = pd.DataFrame()

    # --------------------------------------------------------
    # SUMMARY
    # --------------------------------------------------------

    print("\n" + "=" * 70)
    print(
        f"{dataset_name} — RESUMEN "
        "ANOMALÍAS CONTEXTUALES"
    )
    print("=" * 70)

    print(
        f"Total eventos/filas: "
        f"{len(df_eventos):,}"
    )

    if not df_eventos.empty:

        print("\nPor contexto:")

        print(
            df_eventos[
                "contexto"
            ].value_counts()
        )

        print("\nPor sensor:")

        print(
            df_eventos[
                "sensor_anomalia"
            ].value_counts()
        )

    print(
        f"\nFilas usadas acumuladas: "
        f"{used_rows.sum():,}"
    )

    return (
        df_after_3,
        used_rows,
        df_eventos,
    )


# ============================================================
# 5. APPLY TO TRAIN
# ============================================================

(
    df_train_after_contextual,
    used_rows_train,
    df_eventos_contextual_train,
) = aplicar_anomalias_contextuales(

    df=df_train_after_corr,

    used_rows=used_rows_train,

    dataset_name="TRAIN",

    random_state=RANDOM_STATE,
)


# ============================================================
# 6. APPLY TO VALIDATION
# ============================================================

(
    df_validation_after_contextual,
    used_rows_validation,
    df_eventos_contextual_validation,
) = aplicar_anomalias_contextuales(

    df=df_validation_after_corr,

    used_rows=used_rows_validation,

    dataset_name="VALIDATION",

    random_state=RANDOM_STATE,
)


# ============================================================
# 7. APPLY TO TEST
# ============================================================

(
    df_test_after_contextual,
    used_rows_test,
    df_eventos_contextual_test,
) = aplicar_anomalias_contextuales(

    df=df_test_after_corr,

    used_rows=used_rows_test,

    dataset_name="TEST",

    random_state=RANDOM_STATE,
)


# ============================================================
# 8. COMBINED SUMMARY
# ============================================================

print("\n")
print("#" * 70)
print("RESUMEN FINAL — ANOMALÍAS CONTEXTUALES")
print("#" * 70)

print(
    f"\nTRAIN      : "
    f"{len(df_eventos_contextual_train):,}"
    " eventos/filas"
)

print(
    f"VALIDATION : "
    f"{len(df_eventos_contextual_validation):,}"
    " eventos/filas"
)

print(
    f"TEST       : "
    f"{len(df_eventos_contextual_test):,}"
    " eventos/filas"
)


# ============================================================
# 9. AUDIT FUNCTION
# ============================================================

def audit_contextual(
    df,
    df_eventos,
    used_rows,
    dataset_name,
):

    print("\n" + "=" * 70)
    print(
        f"{dataset_name} — "
        "CONTEXTUAL ANOMALIES AUDIT"
    )
    print("=" * 70)

    if df_eventos is None or df_eventos.empty:

        print(
            "No contextual events to audit."
        )

        return

    # --------------------------------------------------------
    # ROWS WITH CONTEXTUAL TYPE
    # --------------------------------------------------------

    mask_contextual = (
        df[
            "tipo_anomalia"
        ]
        .eq("Contextual")
    )

    df_check = df.loc[
        mask_contextual
    ]

    print(
        f"\nRows Contextual: "
        f"{len(df_check):,}"
    )

    print(
        f"Events registered: "
        f"{len(df_eventos):,}"
    )

    assert len(df_check) == len(
        df_eventos
    ), (
        f"{dataset_name}: "
        "Rows vs events mismatch."
    )

    # --------------------------------------------------------
    # TRACEABILITY
    # --------------------------------------------------------

    metadata_columns = [
        "sensor_anomalia",
        "anomalia_id",
        "tipo_anomalia",
        "fuente_anomalia",
        "valor_original_anomalia",
        "valor_inyectado_anomalia",
    ]

    print("\nTraceability:")

    for col in metadata_columns:

        missing = int(
            df_check[col]
            .isna()
            .sum()
        )

        print(
            f"{col:30s}: "
            f"{missing} missing"
        )

        assert missing == 0

    # --------------------------------------------------------
    # TYPE
    # --------------------------------------------------------

    wrong_type = int(
        df_check[
            "tipo_anomalia"
        ]
        .ne("Contextual")
        .sum()
    )

    print(
        "\nWrong anomaly type:",
        wrong_type
    )

    assert wrong_type == 0

    # --------------------------------------------------------
    # SOURCE
    # --------------------------------------------------------

    wrong_source = int(
        df_check[
            "fuente_anomalia"
        ]
        .ne(
            "professor_method_contextual"
        )
        .sum()
    )

    print(
        "Wrong anomaly source:",
        wrong_source
    )

    assert wrong_source == 0

    # --------------------------------------------------------
    # UNIQUE IDS
    # --------------------------------------------------------

    unique_ids = (
        df_check[
            "anomalia_id"
        ]
        .nunique()
    )

    duplicate_ids = (
        len(df_check)
        -
        unique_ids
    )

    print(
        "\nUnique anomaly IDs:",
        unique_ids
    )

    print(
        "Duplicate IDs:",
        duplicate_ids
    )

    assert duplicate_ids == 0

    # --------------------------------------------------------
    # ORIGINAL != INJECTED
    # --------------------------------------------------------

    same_values = np.isclose(

        df_check[
            "valor_original_anomalia"
        ].astype(float),

        df_check[
            "valor_inyectado_anomalia"
        ].astype(float),

        equal_nan=False,
    )

    same_count = int(
        same_values.sum()
    )

    print(
        "\nOriginal == injected:",
        same_count
    )

    assert same_count == 0

    # --------------------------------------------------------
    # EVENT POSITIONS
    # --------------------------------------------------------

    event_positions = (
        df_eventos[
            "row_position"
        ]
        .astype(int)
        .to_numpy()
    )

    not_marked = int(
        np.sum(
            ~used_rows[
                event_positions
            ]
        )
    )

    duplicated_positions = (
        len(event_positions)
        -
        len(
            np.unique(
                event_positions
            )
        )
    )

    print(
        "Events not marked in used_rows:",
        not_marked
    )

    print(
        "Duplicate positions:",
        duplicated_positions
    )

    assert not_marked == 0

    assert duplicated_positions == 0

    # --------------------------------------------------------
    # CONTEXT COUNTS
    # --------------------------------------------------------

    print("\nBy context:")

    print(
        df_eventos[
            "contexto"
        ].value_counts()
    )

    print("\nBy sensor:")

    print(
        df_eventos[
            "sensor_anomalia"
        ].value_counts()
    )

    # --------------------------------------------------------
    # CONTEXT-SPECIFIC CHECKS
    # --------------------------------------------------------

    # Type 1
    mask_1 = (
        df_eventos[
            "contexto"
        ]
        .eq("PRAD_PRGINT")
    )

    if mask_1.any():

        ratio_values = (
            df_eventos.loc[
                mask_1,
                "ratio_PRGINT_PRAD"
            ]
        )

        assert (
            ratio_values
            <=
            cfg_contextual_prad[
                "ratio_anomalo_max"
            ]
        ).all()

    # Type 2
    mask_2 = (
        df_eventos[
            "contexto"
        ]
        .eq("CO2_VENTILACION")
    )

    if mask_2.any():

        co2_values = (
            df_eventos.loc[
                mask_2,
                "valor_inyectado"
            ]
        )

        assert (
            co2_values
            >=
            cfg_contextual_co2[
                "co2_anomalo_min"
            ] - 10
        ).all()

        assert (
            co2_values
            <=
            cfg_contextual_co2[
                "co2_anomalo_max"
            ] + 10
        ).all()

    # Type 3
    mask_3 = (
        df_eventos[
            "contexto"
        ]
        .eq("TEMP_INVERSION")
    )

    if mask_3.any():

        delta_values = (
            df_eventos.loc[
                mask_3,
                "delta"
            ]
        )

        assert (
            delta_values
            >=
            cfg_contextual_temp[
                "delta_min"
            ]
        ).all()

        assert (
            delta_values
            <=
            cfg_contextual_temp[
                "delta_max"
            ]
        ).all()

    print("\n" + "-" * 70)

    print(
        f"✓ {dataset_name} Contextual audit passed."
    )

    print("-" * 70)


# ============================================================
# 10. RUN AUDITS
# ============================================================

audit_contextual(
    df_train_after_contextual,
    df_eventos_contextual_train,
    used_rows_train,
    "TRAIN",
)

audit_contextual(
    df_validation_after_contextual,
    df_eventos_contextual_validation,
    used_rows_validation,
    "VALIDATION",
)

audit_contextual(
    df_test_after_contextual,
    df_eventos_contextual_test,
    used_rows_test,
    "TEST",
)


# ============================================================
# 11. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("CONTEXTUAL ANOMALIES — FINAL SUMMARY")
print("=" * 70)

print(
    f"TRAIN      : "
    f"{len(df_eventos_contextual_train):,}"
)

print(
    f"VALIDATION : "
    f"{len(df_eventos_contextual_validation):,}"
)

print(
    f"TEST       : "
    f"{len(df_eventos_contextual_test):,}"
)

print("\nOverlap control:")

print(
    f"  TRAIN used rows      : "
    f"{used_rows_train.sum():,}"
)

print(
    f"  VALIDATION used rows : "
    f"{used_rows_validation.sum():,}"
)

print(
    f"  TEST used rows       : "
    f"{used_rows_test.sum():,}"
)

print(
    "\n✓ Contextual anomaly injection "
    "completed for TRAIN / VALIDATION / TEST."
)

print("=" * 70)

In [ ]:
# ============================================================
# FINAL SUMMARY OF INJECTED ANOMALIES
# TRAIN + VALIDATION + TEST
# ============================================================

print("\n" + "#" * 70)
print("RESUMEN FINAL DE ANOMALÍAS INYECTADAS")
print("#" * 70)


# ============================================================
# 1. COLLECT EVENTS BY DATASET
# ============================================================

event_tables = []


# ============================================================
# TRAIN
# ============================================================

train_event_sources = [
    (
        "Sensor Atascado",
        "df_eventos_stuck_train",
    ),
    (
        "Datos Faltantes",
        "df_eventos_missing_train",
    ),
    (
        "Ruido",
        "df_eventos_ruido_train",
    ),
    (
        "Valores Fuera de Rango",
        "df_eventos_oor_train",
    ),
    (
        "Desviación de Correlación",
        "df_eventos_corr_train",
    ),
    (
        "Contextual",
        "df_eventos_contextual_train",
    ),
]


for anomaly_type, variable_name in train_event_sources:

    if variable_name in locals():

        events = locals()[variable_name]

        if events is not None and not events.empty:

            tmp = events.copy()

            tmp["dataset"] = "TRAIN"
            tmp["tipo_resumen"] = anomaly_type

            event_tables.append(tmp)


# ============================================================
# VALIDATION
# ============================================================

validation_event_sources = [
    (
        "Sensor Atascado",
        "df_eventos_stuck_validation",
    ),
    (
        "Datos Faltantes",
        "df_eventos_missing_validation",
    ),
    (
        "Ruido",
        "df_eventos_ruido_validation",
    ),
    (
        "Valores Fuera de Rango",
        "df_eventos_oor_validation",
    ),
    (
        "Desviación de Correlación",
        "df_eventos_corr_validation",
    ),
    (
        "Contextual",
        "df_eventos_contextual_validation",
    ),
]


for anomaly_type, variable_name in validation_event_sources:

    if variable_name in locals():

        events = locals()[variable_name]

        if events is not None and not events.empty:

            tmp = events.copy()

            tmp["dataset"] = "VALIDATION"
            tmp["tipo_resumen"] = anomaly_type

            event_tables.append(tmp)


# ============================================================
# TEST
# ============================================================

test_event_sources = [
    (
        "Sensor Atascado",
        "df_eventos_stuck_test",
    ),
    (
        "Datos Faltantes",
        "df_eventos_missing_test",
    ),
    (
        "Ruido",
        "df_eventos_ruido_test",
    ),
    (
        "Valores Fuera de Rango",
        "df_eventos_oor_test",
    ),
    (
        "Desviación de Correlación",
        "df_eventos_corr_test",
    ),
    (
        "Contextual",
        "df_eventos_contextual_test",
    ),
]


for anomaly_type, variable_name in test_event_sources:

    if variable_name in locals():

        events = locals()[variable_name]

        if events is not None and not events.empty:

            tmp = events.copy()

            tmp["dataset"] = "TEST"
            tmp["tipo_resumen"] = anomaly_type

            event_tables.append(tmp)


# ============================================================
# 2. COMBINE EVENTS
# ============================================================

if event_tables:

    df_eventos_total = pd.concat(
        event_tables,
        ignore_index=True,
        sort=False,
    )

else:

    df_eventos_total = pd.DataFrame()


# ============================================================
# 3. SUMMARY BY DATASET
# ============================================================

print("\n" + "=" * 60)
print("1. ANOMALÍAS POR DATASET")
print("=" * 60)

if not df_eventos_total.empty:

    print(
        df_eventos_total[
            "dataset"
        ].value_counts()
    )

else:

    print("No hay eventos registrados.")


# ============================================================
# 4. SUMMARY BY ANOMALY TYPE
# ============================================================

print("\n" + "=" * 60)
print("2. ANOMALÍAS POR TIPO")
print("=" * 60)

if not df_eventos_total.empty:

    print(
        df_eventos_total[
            "tipo_resumen"
        ].value_counts()
    )

else:

    print("No hay eventos registrados.")


# ============================================================
# 5. TYPE × DATASET
# ============================================================

print("\n" + "=" * 60)
print("3. ANOMALÍAS POR TIPO Y DATASET")
print("=" * 60)

if not df_eventos_total.empty:

    summary_dataset_type = pd.crosstab(
        df_eventos_total["tipo_resumen"],
        df_eventos_total["dataset"],
    )

    print(
        summary_dataset_type
    )

else:

    summary_dataset_type = pd.DataFrame()

    print("No hay datos.")


# ============================================================
# 6. SUMMARY BY SENSOR
# ============================================================

print("\n" + "=" * 60)
print("4. ANOMALÍAS POR SENSOR")
print("=" * 60)

if (
    not df_eventos_total.empty
    and "sensor_anomalia"
    in df_eventos_total.columns
):

    print(
        df_eventos_total[
            "sensor_anomalia"
        ].value_counts()
    )

else:

    print(
        "No existe información de "
        "'sensor_anomalia'."
    )


# ============================================================
# 7. SENSOR × DATASET
# ============================================================

print("\n" + "=" * 60)
print("5. ANOMALÍAS POR SENSOR Y DATASET")
print("=" * 60)

if (
    not df_eventos_total.empty
    and "sensor_anomalia"
    in df_eventos_total.columns
):

    summary_sensor_dataset = pd.crosstab(
        df_eventos_total[
            "sensor_anomalia"
        ],
        df_eventos_total[
            "dataset"
        ],
    )

    print(
        summary_sensor_dataset
    )

else:

    summary_sensor_dataset = pd.DataFrame()

    print("No hay datos.")


# ============================================================
# 8. CONTEXTUAL SUBTYPES
# ============================================================

print("\n" + "=" * 60)
print("6. DESGLOSE DE ANOMALÍAS CONTEXTUALES")
print("=" * 60)

if (
    not df_eventos_total.empty
    and "contexto"
    in df_eventos_total.columns
):

    contextual_events = (
        df_eventos_total[
            df_eventos_total[
                "tipo_resumen"
            ]
            == "Contextual"
        ]
    )

    if not contextual_events.empty:

        print(
            contextual_events[
                "contexto"
            ].value_counts()
        )

    else:

        print(
            "No hay eventos contextuales."
        )

else:

    print(
        "No hay información de "
        "subtipos contextuales."
    )


# ============================================================
# 9. CONTEXTUAL × DATASET
# ============================================================

print("\n" + "=" * 60)
print("7. CONTEXTUALES POR DATASET")
print("=" * 60)

if (
    not contextual_events.empty
    and "contexto"
    in contextual_events.columns
):

    contextual_dataset_summary = pd.crosstab(
        contextual_events[
            "contexto"
        ],
        contextual_events[
            "dataset"
        ],
    )

    print(
        contextual_dataset_summary
    )

else:

    contextual_dataset_summary = pd.DataFrame()

    print(
        "No hay datos contextuales."
    )


# ============================================================
# 10. TOTAL EVENT RECORDS
# ============================================================

print("\n" + "=" * 60)
print("8. TOTAL DE REGISTROS DE EVENTOS")
print("=" * 60)

total_event_records = (
    len(df_eventos_total)
)

print(
    f"Total de registros de eventos: "
    f"{total_event_records:,}"
)


# ============================================================
# 11. UNIQUE AFFECTED ROWS — PER DATASET
# ============================================================

print("\n" + "=" * 60)
print("9. FILAS ÚNICAS AFECTADAS")
print("=" * 60)


unique_positions_by_dataset = {}


if (
    not df_eventos_total.empty
    and "row_position"
    in df_eventos_total.columns
):

    for dataset_name in [
        "TRAIN",
        "VALIDATION",
        "TEST",
    ]:

        subset = df_eventos_total[
            df_eventos_total[
                "dataset"
            ]
            == dataset_name
        ]

        if not subset.empty:

            unique_positions = (
                subset[
                    "row_position"
                ]
                .dropna()
                .astype(int)
                .nunique()
            )

        else:

            unique_positions = 0

        unique_positions_by_dataset[
            dataset_name
        ] = unique_positions

        print(
            f"{dataset_name:12s}: "
            f"{unique_positions:,}"
        )

else:

    print(
        "No se puede calcular "
        "'row_position'."
    )


# ============================================================
# 12. DATASET SIZES
# ============================================================

print("\n" + "=" * 60)
print("10. TAMAÑO DE LOS DATASETS")
print("=" * 60)

dataset_sizes = {
    "TRAIN":
        len(df_train),

    "VALIDATION":
        len(df_validation),

    "TEST":
        len(df_test),
}


for dataset_name, size in dataset_sizes.items():

    print(
        f"{dataset_name:12s}: "
        f"{size:,}"
    )


# ============================================================
# 13. PERCENTAGE OF UNIQUE ROWS AFFECTED
# ============================================================

print("\n" + "=" * 60)
print(
    "11. PORCENTAJE DE FILAS ÚNICAS AFECTADAS"
)
print("=" * 60)

percentage_affected_by_dataset = {}


for dataset_name in [
    "TRAIN",
    "VALIDATION",
    "TEST",
]:

    unique_count = (
        unique_positions_by_dataset.get(
            dataset_name,
            0
        )
    )

    dataset_size = (
        dataset_sizes[
            dataset_name
        ]
    )

    if dataset_size > 0:

        percentage = (
            unique_count
            /
            dataset_size
            *
            100
        )

    else:

        percentage = 0.0

    percentage_affected_by_dataset[
        dataset_name
    ] = percentage

    print(
        f"{dataset_name:12s}: "
        f"{percentage:.2f}%"
    )


# ============================================================
# 14. USED ROWS CHECK
# ============================================================

print("\n" + "=" * 60)
print("12. USED_ROWS POR DATASET")
print("=" * 60)

used_rows_by_dataset = {
    "TRAIN":
        used_rows_train,

    "VALIDATION":
        used_rows_validation,

    "TEST":
        used_rows_test,
}


for dataset_name, mask in (
    used_rows_by_dataset.items()
):

    print(
        f"{dataset_name:12s}: "
        f"{int(mask.sum()):,}"
    )


# ============================================================
# 15. CHECK EVENT POSITIONS
# ============================================================

print("\n" + "=" * 60)
print(
    "13. EVENTOS NO REGISTRADOS EN USED_ROWS"
)
print("=" * 60)

not_marked_by_dataset = {}


for dataset_name in [
    "TRAIN",
    "VALIDATION",
    "TEST",
]:

    subset = df_eventos_total[
        df_eventos_total[
            "dataset"
        ]
        ==
        dataset_name
    ]

    if (
        subset.empty
        or
        "row_position"
        not in subset.columns
    ):

        not_marked = 0

    else:

        valid_positions = (
            subset[
                "row_position"
            ]
            .dropna()
            .astype(int)
            .to_numpy()
        )

        used_mask = (
            used_rows_by_dataset[
                dataset_name
            ]
        )

        # Safety check
        valid_positions = (
            valid_positions[
                (valid_positions >= 0)
                &
                (
                    valid_positions
                    <
                    len(used_mask)
                )
            ]
        )

        not_marked = int(
            np.sum(
                ~used_mask[
                    valid_positions
                ]
            )
        )

    not_marked_by_dataset[
        dataset_name
    ] = not_marked

    print(
        f"{dataset_name:12s}: "
        f"{not_marked}"
    )


# ============================================================
# 16. DUPLICATE POSITIONS — PER DATASET
# ============================================================

print("\n" + "=" * 60)
print(
    "14. POSICIONES DUPLICADAS POR DATASET"
)
print("=" * 60)

duplicate_positions_by_dataset = {}


for dataset_name in [
    "TRAIN",
    "VALIDATION",
    "TEST",
]:

    subset = df_eventos_total[
        df_eventos_total[
            "dataset"
        ]
        ==
        dataset_name
    ]

    if (
        subset.empty
        or
        "row_position"
        not in subset.columns
    ):

        duplicate_positions = 0

    else:

        positions = (
            subset[
                "row_position"
            ]
            .dropna()
            .astype(int)
        )

        duplicate_positions = (
            len(positions)
            -
            positions.nunique()
        )

    duplicate_positions_by_dataset[
        dataset_name
    ] = duplicate_positions

    print(
        f"{dataset_name:12s}: "
        f"{duplicate_positions}"
    )


# ============================================================
# 17. FINAL SUMMARY TABLE
# ============================================================

print("\n" + "#" * 70)
print("TABLA RESUMEN FINAL")
print("#" * 70)

if not df_eventos_total.empty:

    summary_by_type_dataset = (
        df_eventos_total
        .groupby(
            [
                "dataset",
                "tipo_resumen",
            ]
        )
        .size()
        .unstack(
            fill_value=0
        )
    )

    print(
        summary_by_type_dataset
    )

else:

    summary_by_type_dataset = (
        pd.DataFrame()
    )

    print("No hay datos.")


# ============================================================
# 18. FINAL AUDIT
# ============================================================

print("\n" + "#" * 70)
print("AUDITORÍA FINAL")
print("#" * 70)


# ------------------------------------------------------------
# USED_ROWS CHECK
# ------------------------------------------------------------

all_used_rows_ok = all(
    value == 0
    for value in not_marked_by_dataset.values()
)

if all_used_rows_ok:

    print(
        "✓ Todos los eventos están registrados "
        "en used_rows."
    )

else:

    print(
        "⚠ Hay eventos no registrados "
        "en used_rows."
    )


# ------------------------------------------------------------
# DUPLICATE POSITIONS CHECK
# ------------------------------------------------------------

all_positions_unique = all(
    value == 0
    for value in duplicate_positions_by_dataset.values()
)

if all_positions_unique:

    print(
        "✓ No existen posiciones duplicadas "
        "dentro de ningún dataset."
    )

else:

    print(
        "⚠ Existen posiciones duplicadas."
    )



# ------------------------------------------------------------
# FINAL
# ------------------------------------------------------------

if (
    all_used_rows_ok
    and all_positions_unique
    and event_count_ok
):

    print(
        "\n✓ Auditoría final completada "
        "correctamente."
    )

else:

    print(
        "\n⚠ Revisar las advertencias "
        "anteriores."
    )


print("\n" + "#" * 70)
print(
    "RESUMEN FINAL DE ANOMALÍAS "
    "COMPLETADO"
)
print("#" * 70)

In [ ]:
# ============================================================
# FINAL INTEGRITY CHECK + GUARDADO
#
# Final outputs:
#   data/injected/train_injected_physical.parquet
#   data/injected/validation_injected_physical.parquet
#   data/injected/test_injected_physical.parquet
#
# Files contain:
#   - physical/unscaled sensor values
#   - FECHA
#   - anomaly traceability metadata
#
# Clean dataframes are NOT overwritten.
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np


# ============================================================
# 1. FINAL INJECTED DATAFRAMES
# ============================================================

df_train_injected = (
    df_train_after_contextual.copy()
)

df_validation_injected = (
    df_validation_after_contextual.copy()
)

df_test_injected = (
    df_test_after_contextual.copy()
)


# ============================================================
# 2. ENSURE FECHA IS A COLUMN
# ============================================================

for name in [
    "df_train_injected",
    "df_validation_injected",
    "df_test_injected",
]:

    df_current = locals()[name]

    if (
        isinstance(
            df_current.index,
            pd.DatetimeIndex
        )
        and df_current.index.name == "FECHA"
    ):

        df_current = (
            df_current.reset_index()
        )

        locals()[name] = df_current


# ============================================================
# 3. DATASET CONFIGURATION
# ============================================================

datasets = {

    "TRAIN": {
        "df": df_train_injected,
        "clean": df_train,
        "used_rows": used_rows_train,
    },

    "VALIDATION": {
        "df": df_validation_injected,
        "clean": df_validation,
        "used_rows": used_rows_validation,
    },

    "TEST": {
        "df": df_test_injected,
        "clean": df_test,
        "used_rows": used_rows_test,
    },
}


# ============================================================
# 4. OUTPUT DIRECTORY
# ============================================================

DATA_INJECTED_DIR = Path(
    "/home/jetson/Desktop/ciesol-anomaly-detection/data/injected"
)

DATA_INJECTED_DIR.mkdir(
    parents=True,
    exist_ok=True
)


output_paths = {

    "TRAIN":
        DATA_INJECTED_DIR /
        "train_injected_physical.parquet",

    "VALIDATION":
        DATA_INJECTED_DIR /
        "validation_injected_physical.parquet",

    "TEST":
        DATA_INJECTED_DIR /
        "test_injected_physical.parquet",
}


# ============================================================
# 5. TRACEABILITY COLUMNS
# ============================================================

trace_columns = [
    "tipo_anomalia",
    "sensor_anomalia",
]


# ============================================================
# 6. INTEGRITY CHECK FUNCTION
# ============================================================

def check_injected_dataset(
    dataset_name,
    df_injected,
    df_clean,
    used_rows,
):

    print("\n" + "=" * 70)
    print(
        f"FINAL INTEGRITY CHECK — "
        f"INJECTED {dataset_name}"
    )
    print("=" * 70)


    # --------------------------------------------------------
    # Basic structure
    # --------------------------------------------------------

    print(
        f"\nClean {dataset_name} rows:    "
        f"{len(df_clean):,}"
    )

    print(
        f"Injected {dataset_name} rows: "
        f"{len(df_injected):,}"
    )

    assert len(df_injected) == len(df_clean), (
        f"ERROR: el número de filas cambió "
        f"en {dataset_name}."
    )

    assert "FECHA" in df_injected.columns, (
        f"ERROR: FECHA no está disponible "
        f"en {dataset_name}."
    )

    print("✓ Número de filas conservado.")
    print("✓ FECHA disponible como columna.")


    # --------------------------------------------------------
    # Sensors
    # --------------------------------------------------------

    missing_sensors = [
        sensor
        for sensor in SENSORES_INYECCION
        if sensor not in df_injected.columns
    ]

    assert not missing_sensors, (
        f"ERROR: faltan sensores de inyección "
        f"en {dataset_name}: "
        f"{missing_sensors}"
    )

    print(
        "✓ Todos los sensores de inyección "
        "están presentes."
    )


    # --------------------------------------------------------
    # Traceability
    # --------------------------------------------------------

    for col in trace_columns:

        assert col in df_injected.columns, (
            f"ERROR: falta columna de trazabilidad "
            f"'{col}' en {dataset_name}."
        )

    print(
        "✓ Columnas de trazabilidad disponibles."
    )


    # --------------------------------------------------------
    # Anomaly rows
    # --------------------------------------------------------

    mask_anomaly = (
        df_injected[
            "tipo_anomalia"
        ]
        .fillna("")
        .astype(str)
        .ne("")
    )

    n_anomaly_rows = int(
        mask_anomaly.sum()
    )

    n_rows = len(df_injected)

    percentage = (
        100 * n_anomaly_rows / n_rows
        if n_rows > 0
        else 0
    )

    print(
        f"\nFilas con alguna anomalía: "
        f"{n_anomaly_rows:,}"
    )

    print(
        f"Porcentaje afectado: "
        f"{percentage:.2f}%"
    )


    # --------------------------------------------------------
    # Anomalies by type
    # --------------------------------------------------------

    print("\n" + "-" * 70)
    print("ANOMALÍAS POR TIPO")
    print("-" * 70)

    anomaly_by_type = (
        df_injected.loc[
            mask_anomaly,
            "tipo_anomalia"
        ]
        .value_counts()
    )

    if not anomaly_by_type.empty:

        print(
            anomaly_by_type.to_string()
        )

    else:

        print("No hay anomalías registradas.")


    # --------------------------------------------------------
    # Anomalies by sensor
    # --------------------------------------------------------

    print("\n" + "-" * 70)
    print("ANOMALÍAS POR SENSOR")
    print("-" * 70)

    if "sensor_anomalia" in df_injected.columns:

        sensor_values = (
            df_injected.loc[
                mask_anomaly,
                "sensor_anomalia"
            ]
            .dropna()
            .astype(str)
        )

        sensor_counts = {}

        for value in sensor_values:

            sensors = [
                s.strip()
                for s in value.split(",")
                if s.strip()
            ]

            for sensor in sensors:

                sensor_counts[sensor] = (
                    sensor_counts.get(
                        sensor,
                        0
                    ) + 1
                )

        if sensor_counts:

            sensor_summary = (
                pd.Series(sensor_counts)
                .sort_values(
                    ascending=False
                )
            )

            print(
                sensor_summary.to_string()
            )

        else:

            print(
                "No hay sensores registrados."
            )


    # --------------------------------------------------------
    # used_rows
    # --------------------------------------------------------

    print("\n" + "-" * 70)
    print("CONTROL DE used_rows")
    print("-" * 70)

    used_array = np.asarray(
        used_rows,
        dtype=bool
    )

    assert len(used_array) == len(
        df_injected
    ), (
        f"ERROR: used_rows tiene longitud "
        f"{len(used_array):,}, pero "
        f"{dataset_name} tiene "
        f"{len(df_injected):,} filas."
    )

    used_count = int(
        used_array.sum()
    )

    print(
        f"Filas marcadas como usadas: "
        f"{used_count:,}"
    )


    # --------------------------------------------------------
    # used_rows
    # --------------------------------------------------------

    print("\n" + "-" * 70)
    print("CONTROL DE used_rows")
    print("-" * 70)

    used_array = np.asarray(
        used_rows,
        dtype=bool
    )

    assert len(used_array) == len(
        df_injected
    ), (
        f"ERROR: used_rows tiene longitud "
        f"{len(used_array):,}, pero "
        f"{dataset_name} tiene "
        f"{len(df_injected):,} filas."
    )

    used_count = int(
        used_array.sum()
    )

    print(
        f"Filas marcadas como usadas: "
        f"{used_count:,}"
    )

    print(
        "✓ used_rows tiene la longitud "
        "correcta."
    )

    # --------------------------------------------------------
    # No scaled columns
    # --------------------------------------------------------

    scaled_columns = [
        c
        for c in df_injected.columns
        if c.endswith("_scaled")
    ]

    print("\n" + "-" * 70)
    print("CONTROL DE ESCALADO")
    print("-" * 70)

    print(
        f"Columnas *_scaled encontradas: "
        f"{len(scaled_columns)}"
    )

    assert len(scaled_columns) == 0, (
        f"ERROR: {dataset_name} contiene "
        "columnas *_scaled."
    )

    print(
        "✓ Valores físicos/no escalados."
    )


    # --------------------------------------------------------
    # No duplicate index positions
    # --------------------------------------------------------

    assert not df_injected.index.duplicated().any(), (
        f"ERROR: existen índices duplicados "
        f"en {dataset_name}."
    )

    print(
        "✓ No existen índices duplicados."
    )


    # --------------------------------------------------------
    # Return summary
    # --------------------------------------------------------

    return {
        "dataset": dataset_name,
        "rows": len(df_injected),
        "anomaly_rows": n_anomaly_rows,
        "percentage_affected": percentage,
        "used_rows": used_count,
    }


# ============================================================
# 7. RUN FINAL CHECKS
# ============================================================

final_checks = []

for dataset_name, config in datasets.items():

    result = check_injected_dataset(
        dataset_name=dataset_name,
        df_injected=config["df"],
        df_clean=config["clean"],
        used_rows=config["used_rows"],
    )

    final_checks.append(result)


# ============================================================
# 8. SAVE ALL THREE DATASETS
# ============================================================

print("\n" + "#" * 70)
print("GUARDANDO DATASETS INYECTADOS")
print("#" * 70)


for dataset_name, config in datasets.items():

    path = output_paths[
        dataset_name
    ]

    config["df"].to_parquet(
        path,
        index=False
    )

    print(
        f"✓ {dataset_name}: "
        f"{path}"
    )


# ============================================================
# 9. FINAL SUMMARY
# ============================================================

df_final_checks = pd.DataFrame(
    final_checks
)

print("\n" + "#" * 70)
print("RESUMEN FINAL DE GUARDADO")
print("#" * 70)

print(
    df_final_checks.to_string(
        index=False
    )
)


# ============================================================
# 10. FINAL CONFIRMATION
# ============================================================

print("\n" + "#" * 70)
print("GUARDADO COMPLETADO")
print("#" * 70)

print(
    "✓ TRAIN guardado por separado."
)

print(
    "✓ VALIDATION guardado por separado."
)

print(
    "✓ TEST guardado por separado."
)

print(
    "✓ Los datasets clean NO fueron sobrescritos."
)

print(
    "✓ Todos los valores siguen en escala física."
)

print(
    "✓ La trazabilidad de anomalías está conservada."
)

print(
    "✓ used_rows validado independientemente "
    "para TRAIN / VALIDATION / TEST."
)

print(
    "\n✓ INTEGRIDAD FINAL: PASS"
)

In [ ]:
# ============================================================
# COMPREHENSIVE FINAL VALIDATION
# TRAIN / VALIDATION / TEST
#
# This cell ONLY checks the data.
# It does NOT modify the dataframes.
# ============================================================

import numpy as np
import pandas as pd


# ============================================================
# 1. DATASETS TO CHECK
# ============================================================

datasets_to_check = {
    "TRAIN": {
        "df": df_train_after_contextual,
        "clean": df_train,
        "used_rows": used_rows_train,
    },
    "VALIDATION": {
        "df": df_validation_after_contextual,
        "clean": df_validation,
        "used_rows": used_rows_validation,
    },
    "TEST": {
        "df": df_test_after_contextual,
        "clean": df_test,
        "used_rows": used_rows_test,
    },
}


# ============================================================
# 2. EXPECTED COLUMNS
# ============================================================

EXPECTED_SENSOR_COLUMNS = [
    "PCO2EXT",
    "PHEXT",
    "PRAD",
    "PRGINT",
    "PTEXT",
    "PVV",
    "UVENT_cen",
    "UVENT_lN",
    "XCO2I",
    "XHINV",
    "XTINV",
    "XTS",
]

EXPECTED_TRACE_COLUMNS = [
    "sensor_anomalia",
    "anomalia_id",
    "tipo_anomalia",
    "fuente_anomalia",
    "valor_original_anomalia",
    "valor_inyectado_anomalia",
]

EXPECTED_QUALITY_COLUMNS = [
    "timestamp_was_missing",
    "_original_observation",
]


# ============================================================
# 3. PHYSICAL RANGES
# ============================================================

# Uses your canonical RANGOS_FISICOS if available.

if "RANGOS_FISICOS" not in locals():

    RANGOS_FISICOS = {
        "PCO2EXT": {"min": 300, "max": 800},
        "PHEXT": {"min": 0, "max": 100},
        "PRAD": {"min": 0, "max": 1500},
        "PRGINT": {"min": 0, "max": 1200},
        "PTEXT": {"min": -20, "max": 60},
        "PVV": {"min": 0, "max": 50},
        "XCO2I": {"min": 200, "max": 3000},
        "XHINV": {"min": 0, "max": 100},
        "XTINV": {"min": -5, "max": 60},
    }


# ============================================================
# 4. RESULTS STORAGE
# ============================================================

validation_results = []


# ============================================================
# 5. VALIDATION FUNCTION
# ============================================================

def validate_final_dataset(
    dataset_name,
    df,
    clean_df,
    used_rows,
):

    print("\n")
    print("=" * 80)
    print(f"VALIDATING FINAL {dataset_name}")
    print("=" * 80)

    errors = []
    warnings = []

    # --------------------------------------------------------
    # A. BASIC SHAPE
    # --------------------------------------------------------

    print("\n[A] BASIC STRUCTURE")

    if len(df) != len(clean_df):

        errors.append(
            f"Row count changed: "
            f"clean={len(clean_df)}, "
            f"final={len(df)}"
        )

    else:

        print(
            f"✓ Row count: {len(df):,}"
        )


    # --------------------------------------------------------
    # B. REQUIRED COLUMNS
    # --------------------------------------------------------

    print("\n[B] REQUIRED COLUMNS")

    required_columns = (
        ["FECHA"]
        + EXPECTED_SENSOR_COLUMNS
        + EXPECTED_TRACE_COLUMNS
        + EXPECTED_QUALITY_COLUMNS
    )

    missing_columns = [
        c
        for c in required_columns
        if c not in df.columns
    ]

    if missing_columns:

        errors.append(
            f"Missing columns: {missing_columns}"
        )

        print(
            f"✗ Missing columns: "
            f"{missing_columns}"
        )

    else:

        print(
            f"✓ All {len(required_columns)} "
            "required columns exist."
        )


    # --------------------------------------------------------
    # C. UNEXPECTED DUPLICATE COLUMNS
    # --------------------------------------------------------

    print("\n[C] COLUMN DUPLICATES")

    duplicated_columns = (
        df.columns[
            df.columns.duplicated()
        ]
        .tolist()
    )

    if duplicated_columns:

        errors.append(
            f"Duplicate columns: "
            f"{duplicated_columns}"
        )

        print(
            f"✗ Duplicate columns: "
            f"{duplicated_columns}"
        )

    else:

        print(
            "✓ No duplicate column names."
        )


    # --------------------------------------------------------
    # D. FECHA
    # --------------------------------------------------------

    print("\n[D] TIMESTAMP")

    if "FECHA" in df.columns:

        fecha = pd.to_datetime(
            df["FECHA"],
            errors="coerce"
        )

        invalid_fecha = int(
            fecha.isna().sum()
        )

        if invalid_fecha:

            errors.append(
                f"Invalid FECHA values: "
                f"{invalid_fecha}"
            )

            print(
                f"✗ Invalid timestamps: "
                f"{invalid_fecha:,}"
            )

        else:

            print(
                "✓ All FECHA values are valid."
            )

        duplicate_fecha = int(
            fecha.duplicated().sum()
        )

        if duplicate_fecha:

            errors.append(
                f"Duplicate timestamps: "
                f"{duplicate_fecha}"
            )

            print(
                f"✗ Duplicate timestamps: "
                f"{duplicate_fecha:,}"
            )

        else:

            print(
                "✓ No duplicate timestamps."
            )

        if not fecha.empty:

            print(
                f"  Start: {fecha.min()}"
            )

            print(
                f"  End:   {fecha.max()}"
            )


    # --------------------------------------------------------
    # E. SENSOR DTYPES
    # --------------------------------------------------------

    print("\n[E] SENSOR DATA TYPES")

    bad_dtypes = []

    for sensor in EXPECTED_SENSOR_COLUMNS:

        if sensor not in df.columns:
            continue

        if not pd.api.types.is_numeric_dtype(
            df[sensor]
        ):

            bad_dtypes.append(
                (
                    sensor,
                    str(df[sensor].dtype)
                )
            )

    if bad_dtypes:

        errors.append(
            f"Non-numeric sensors: "
            f"{bad_dtypes}"
        )

        print(
            f"✗ Non-numeric sensors: "
            f"{bad_dtypes}"
        )

    else:

        print(
            "✓ All sensor columns are numeric."
        )


    # --------------------------------------------------------
    # F. NaN CHECK
    # --------------------------------------------------------

    print("\n[F] NaN / MISSING VALUES")

    nan_summary = (
        df[
            EXPECTED_SENSOR_COLUMNS
        ]
        .isna()
        .sum()
    )

    total_sensor_nan = int(
        nan_summary.sum()
    )

    print(
        f"Total sensor NaN values: "
        f"{total_sensor_nan:,}"
    )

    if total_sensor_nan > 0:

        print(
            "\nNaN by sensor:"
        )

        print(
            nan_summary[
                nan_summary > 0
            ]
            .sort_values(
                ascending=False
            )
            .to_string()
        )

        # This is a warning, not an error,
        # because Datos Faltantes are intentionally injected.

        warnings.append(
            f"{total_sensor_nan:,} sensor NaN values "
            "exist. Check that they are intentional."
        )

    else:

        print(
            "✓ No sensor NaN values."
        )


    # --------------------------------------------------------
    # G. INFINITE VALUES
    # --------------------------------------------------------

    print("\n[G] INFINITE VALUES")

    infinite_counts = {}

    for sensor in EXPECTED_SENSOR_COLUMNS:

        if sensor not in df.columns:
            continue

        values = pd.to_numeric(
            df[sensor],
            errors="coerce"
        )

        count_inf = int(
            np.isinf(values).sum()
        )

        if count_inf > 0:

            infinite_counts[
                sensor
            ] = count_inf

    if infinite_counts:

        errors.append(
            f"Infinite values: "
            f"{infinite_counts}"
        )

        print(
            f"✗ Infinite values found: "
            f"{infinite_counts}"
        )

    else:

        print(
            "✓ No infinite sensor values."
        )


    # --------------------------------------------------------
    # H. PHYSICAL RANGE CHECK
    # --------------------------------------------------------

    print("\n[H] PHYSICAL RANGES")

    out_of_range = {}

    for sensor, limits in RANGOS_FISICOS.items():

        if sensor not in df.columns:
            continue

        values = pd.to_numeric(
            df[sensor],
            errors="coerce"
        )

        valid_values = values.dropna()

        if valid_values.empty:
            continue

        n_out = int(
            (
                (valid_values < limits["min"])
                |
                (valid_values > limits["max"])
            ).sum()
        )

        if n_out > 0:

            out_of_range[
                sensor
            ] = n_out

    if out_of_range:

        print(
            "⚠ Values outside physical ranges:"
        )

        print(
            pd.Series(
                out_of_range
            ).to_string()
        )

        # IMPORTANT:
        # OOR anomalies are intentionally outside
        # the normal physical range.

        warnings.append(
            "Out-of-range values exist intentionally "
            "because Valores Fuera de Rango was injected."
        )

    else:

        print(
            "✓ No values outside physical ranges."
        )


    # --------------------------------------------------------
    # I. TRACEABILITY COMPLETENESS
    # --------------------------------------------------------

    print("\n[I] ANOMALY TRACEABILITY")

    anomaly_mask = (
        df[
            "tipo_anomalia"
        ]
        .fillna("")
        .astype(str)
        .ne("")
    )

    n_anomaly_rows = int(
        anomaly_mask.sum()
    )

    print(
        f"Rows marked as anomalous: "
        f"{n_anomaly_rows:,}"
    )

    for col in EXPECTED_TRACE_COLUMNS:

        missing_trace = int(
            df.loc[
                anomaly_mask,
                col
            ]
            .isna()
            .sum()
        )

        if missing_trace > 0:

            warnings.append(
                f"{col}: "
                f"{missing_trace} anomalous rows "
                "have NaN traceability."
            )

            print(
                f"⚠ {col}: "
                f"{missing_trace:,} missing"
            )

        else:

            print(
                f"✓ {col}: complete for "
                "anomalous rows."
            )


    # --------------------------------------------------------
    # J. ANOMALY IDs
    # --------------------------------------------------------

    print("\n[J] ANOMALY IDs")

    if "anomalia_id" in df.columns:

        anomaly_ids = (
            df.loc[
                anomaly_mask,
                "anomalia_id"
            ]
            .dropna()
            .astype(str)
        )

        duplicate_ids = int(
            anomaly_ids.duplicated().sum()
        )

        if duplicate_ids:

            errors.append(
                f"Duplicate anomaly IDs: "
                f"{duplicate_ids}"
            )

            print(
                f"✗ Duplicate IDs: "
                f"{duplicate_ids:,}"
            )

        else:

            print(
                "✓ No duplicate anomaly IDs."
            )


    # --------------------------------------------------------
    # K. ANOMALY TYPES
    # --------------------------------------------------------

    print("\n[K] ANOMALY TYPES")

    if n_anomaly_rows > 0:

        print(
            df.loc[
                anomaly_mask,
                "tipo_anomalia"
            ]
            .value_counts()
            .to_string()
        )


    # --------------------------------------------------------
    # L. USED_ROWS
    # --------------------------------------------------------

    print("\n[L] USED_ROWS")

    used_array = np.asarray(
        used_rows,
        dtype=bool
    )

    if len(used_array) != len(df):

        errors.append(
            f"used_rows length "
            f"{len(used_array):,} != "
            f"dataset length "
            f"{len(df):,}"
        )

        print(
            "✗ used_rows length mismatch."
        )

    else:

        used_count = int(
            used_array.sum()
        )

        print(
            f"✓ used_rows length correct."
        )

        print(
            f"  Rows marked used: "
            f"{used_count:,}"
        )


    # --------------------------------------------------------
    # M. SCALED COLUMNS
    # --------------------------------------------------------

    print("\n[M] SCALING")

    scaled_columns = [
        c
        for c in df.columns
        if c.endswith("_scaled")
    ]

    if scaled_columns:

        errors.append(
            f"Scaled columns found: "
            f"{scaled_columns}"
        )

        print(
            f"✗ *_scaled columns found: "
            f"{scaled_columns}"
        )

    else:

        print(
            "✓ No *_scaled columns."
        )


    # --------------------------------------------------------
    # N. ORIGINAL DATA INTEGRITY
    # --------------------------------------------------------

    print("\n[N] CLEAN DATA INTEGRITY")

    # The anomaly pipeline must not change the number
    # of clean rows.

    if len(df) != len(clean_df):

        errors.append(
            "Final dataset does not have "
            "same number of rows as clean dataset."
        )

    # Check timestamp equality.

    if (
        "FECHA" in df.columns
        and "FECHA" in clean_df.columns
    ):

        fecha_final = pd.to_datetime(
            df["FECHA"],
            errors="coerce"
        )

        fecha_clean = pd.to_datetime(
            clean_df["FECHA"],
            errors="coerce"
        )

        if not fecha_final.equals(
            fecha_clean
        ):

            errors.append(
                "FECHA changed during injection."
            )

            print(
                "✗ FECHA differs from clean data."
            )

        else:

            print(
                "✓ FECHA unchanged."
            )


    # --------------------------------------------------------
    # O. FINAL DATASET RESULT
    # --------------------------------------------------------

    print("\n" + "-" * 70)

    if errors:

        print(
            f"✗ {dataset_name}: "
            f"FAILED"
        )

        print("\nERRORS:")

        for error in errors:

            print(
                f"  ✗ {error}"
            )

    else:

        print(
            f"✓ {dataset_name}: "
            f"PASS"
        )

    if warnings:

        print("\nWARNINGS:")

        for warning in warnings:

            print(
                f"  ⚠ {warning}"
            )

    return {
        "dataset": dataset_name,
        "status": (
            "PASS"
            if not errors
            else "FAIL"
        ),
        "errors": len(errors),
        "warnings": len(warnings),
        "rows": len(df),
        "anomaly_rows": n_anomaly_rows,
        "used_rows": int(
            used_array.sum()
        ),
    }


# ============================================================
# 6. RUN VALIDATION
# ============================================================

validation_results = []

for dataset_name, config in (
    datasets_to_check.items()
):

    result = validate_final_dataset(
        dataset_name=dataset_name,
        df=config["df"],
        clean_df=config["clean"],
        used_rows=config["used_rows"],
    )

    validation_results.append(
        result
    )


# ============================================================
# 7. FINAL VALIDATION REPORT
# ============================================================

print("\n")
print("#" * 80)
print("FINAL VALIDATION REPORT")
print("#" * 80)

validation_report = pd.DataFrame(
    validation_results
)

print(
    validation_report.to_string(
        index=False
    )
)


# ============================================================
# 8. GLOBAL RESULT
# ============================================================

failed_datasets = validation_report[
    validation_report["status"]
    == "FAIL"
]

print("\n" + "#" * 80)

if failed_datasets.empty:

    print(
        "✓ GLOBAL VALIDATION: PASS"
    )

    print(
        "✓ TRAIN / VALIDATION / TEST "
        "are structurally ready."
    )

else:

    print(
        "✗ GLOBAL VALIDATION: FAIL"
    )

    print(
        "Datasets requiring attention:"
    )

    print(
        failed_datasets[
            "dataset"
        ]
        .tolist()
    )

print("#" * 80)

In [ ]:
def validate_anomaly_id_consistency(df, dataset_name):
    """
    Validate anomaly IDs at EVENT level rather than requiring
    one unique ID per affected row.
    """

    print("\n" + "-" * 70)
    print(f"ANOMALY ID CONSISTENCY — {dataset_name}")
    print("-" * 70)

    anomaly_mask = df["tipo_anomalia"].notna()

    anomaly_df = df.loc[
        anomaly_mask,
        [
            "anomalia_id",
            "tipo_anomalia",
            "sensor_anomalia",
            "fuente_anomalia",
        ]
    ].copy()

    # --------------------------------------------------------------
    # 1. Every anomaly row must have an ID
    # --------------------------------------------------------------
    missing_id = anomaly_df["anomalia_id"].isna().sum()

    print(f"Anomaly rows without an ID: {missing_id:,}")

    assert missing_id == 0, (
        f"{dataset_name}: anomaly rows without anomalia_id"
    )

    # --------------------------------------------------------------
    # 2. One ID must correspond to one anomaly type
    # --------------------------------------------------------------
    type_counts = (
        anomaly_df
        .groupby("anomalia_id")["tipo_anomalia"]
        .nunique()
    )

    conflicting_types = type_counts[type_counts > 1]

    print(
        f"IDs associated with multiple anomaly types: "
        f"{len(conflicting_types):,}"
    )

    assert len(conflicting_types) == 0, (
        f"{dataset_name}: some anomaly IDs have multiple types"
    )

    # --------------------------------------------------------------
    # 3. One ID must correspond to one sensor
    # --------------------------------------------------------------
    sensor_counts = (
        anomaly_df
        .groupby("anomalia_id")["sensor_anomalia"]
        .nunique()
    )

    conflicting_sensors = sensor_counts[sensor_counts > 1]

    print(
        f"IDs associated with multiple sensors: "
        f"{len(conflicting_sensors):,}"
    )

    assert len(conflicting_sensors) == 0, (
        f"{dataset_name}: some anomaly IDs have multiple sensors"
    )

    # --------------------------------------------------------------
    # 4. One ID must correspond to one source
    # --------------------------------------------------------------
    source_counts = (
        anomaly_df
        .groupby("anomalia_id")["fuente_anomalia"]
        .nunique()
    )

    conflicting_sources = source_counts[source_counts > 1]

    print(
        f"IDs associated with multiple sources: "
        f"{len(conflicting_sources):,}"
    )

    assert len(conflicting_sources) == 0, (
        f"{dataset_name}: some anomaly IDs have multiple sources"
    )

    # --------------------------------------------------------------
    # 5. Report event count vs affected rows
    # --------------------------------------------------------------
    n_events = anomaly_df["anomalia_id"].nunique()
    n_rows = len(anomaly_df)

    print(f"Unique anomaly events: {n_events:,}")
    print(f"Anomaly rows:          {n_rows:,}")

    print(
        "✓ Duplicate IDs across affected rows are allowed "
        "because IDs identify events."
    )

    print("✓ Anomaly ID consistency PASSED.")

    return True

In [ ]:
for dataset_name, df in {
    "TRAIN": df_train_after_contextual,
    "VALIDATION": df_validation_after_contextual,
    "TEST": df_test_after_contextual,
}.items():

    validate_anomaly_id_consistency(
        df=df,
        dataset_name=dataset_name,
    )